In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

import statsmodels.api as sm
from statsmodels.formula.api import ols, glm

import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from scipy import stats
import shutil

from numpy.polynomial.polynomial import polyfit

# Prepare data

In [ ]:
prayer_df = pd.read_csv('../input/Wellbeing_Results_all.csv')
df_rel_raw = pd.read_csv('../input/areas_dominant_religion_with_reach.csv', sep = ';')
control_vars = gpd.read_parquet('../midsave/control_vars.parquet')

In [ ]:
control_vars.spend.sum()

In [ ]:
wellbeing_1 = (pd.read_csv('../input/Wellbeing_Results_1st_round.csv', sep=";")
               .rename(columns = {'Total_Poll_Responses':'Total_Poll', 'Yes_Counts':'Yes', 'No_Counts':'No'}))
wellbeing_2 = pd.read_csv('../input/Wellbeing_Results_2nd_round.csv', sep=";")

In [ ]:
fb_df = pd.concat([prayer_df.loc[prayer_df.Day.isna(), ['ID', 'Reach', 'Total_Poll', 'Yes', 'No']], 
                    wellbeing_1[['ID', 'Reach', 'Total_Poll', 'Yes', 'No']], 
                    wellbeing_2[['ID', 'Reach', 'Total_Poll', 'Yes', 'No']]], 
                   ignore_index = True).replace(',', '', regex=True).astype(int).copy()

In [ ]:
df_raw = fb_df.merge(control_vars[['ID', 'Location', 'Day', 'Time', 'Gender', 'Age', 'Religion']], on = 'ID', how = 'left')

In [ ]:
df_raw = df_raw[(df_raw.Location.notna())].drop_duplicates(subset = 'ID') # & (df_raw.Reach != 0)

Define relevant controls

In [ ]:
weather_controls = ['temperature_2m',
                    'precipitation',
                    'cloud_cover',
                    'wind_speed_10m',
                    'relative_humidity_2m',#'surface_pressure','sunshine_duration',
                    #'cloud_cover_high','cloud_cover_mid','cloud_cover_low','wind_gusts_10m',
                    #'soil_temperature_0_to_7cm','soil_moisture_0_to_7cm'
                   ]

pop_controls = ['pop_dens_2025',
                #'f_2025','m_2025','pop_2025'
               ]

eco_controls = ['nl_mean_wght','subnational_gdi','sedac_mean', 
                'sedac_var',
                #'nl_mean',
                #'subnational_hdi_females','subnational_hdi_males',
                #'health_index_females','health_index_males',
                #'educational_index_females','educational_index_males',
                #'income_index_females','income_index_males'
               ]

internet_controls = ['internet_speed','fb_pntr_18p_male_tess','fb_pntr_18p_female_tess','ios_age_13_plus_female_frac_2024',
                     'ios_age_13_plus_male_frac_2024','wifi_age_13_plus_female_frac_2024','wifi_age_13_plus_male_frac_2024',
                     'x4g_age_13_plus_female_frac_2024','x4g_age_13_plus_male_frac_2024']

timing_controls = ['hour_of_day_avg', 'day_of_week_start']
                     

In [ ]:
df_pray_raw = (df_raw
               .loc[df_raw.Day.isna(), ['ID', 'Location', 'Religion', 'Gender', 'Reach', 'Total_Poll', 'Yes', 'No']]
               .rename(columns = {'Gender':'Video_Style', 'Reach':'Reach_Pray', 'Yes':'Yes_Pray', 'No':'No_Pray'})
               .reset_index(drop = True))

In [ ]:
df_pray_raw['response_rate'] = df_pray_raw['Total_Poll'] / df_pray_raw['Reach_Pray']*100
df_pray_raw['prayer_ratio'] = df_pray_raw['Yes_Pray'] / (df_pray_raw['Yes_Pray'] + df_pray_raw['No_Pray'])

In [ ]:
df_pray_raw['response_rate'].mean()

In [ ]:
df_pray_raw['Total_Poll'].sum()

In [ ]:
df_well_raw = (df_raw
               .loc[~df_raw.Day.isna()]
               .rename(columns = {'Reach':'Reach_Well', 'Yes':'Yes_Well', 'No':'No_Well'}))

In [ ]:
df_pray_raw[['Location', 'Religion']].drop_duplicates().sort_values(by = "Religion")

# Video response dataset

- Here, we are interested whether the video-style has an impact on the survey response rate from our A/B test
- We control for the dominant religion and for the prayer ratio (as a proxy for religiosity)

In [ ]:
control_video = (control_vars
                    .loc[control_vars.Day.isna(), 
                    ['Location'] + 
                    weather_controls + 
                    pop_controls + 
                    eco_controls + 
                   internet_controls
                    ])

In [ ]:
df_video = (df_pray_raw[['Location','response_rate','Video_Style','Religion','prayer_ratio']]
                   .merge(control_video.drop_duplicates(), on = 'Location', how = 'left')
           )

# Wellbeing Dataset

In [ ]:
df_pray = df_pray_raw.groupby(['Location'])[['Reach_Pray', 'Yes_Pray', 'No_Pray']].sum().reset_index()
df_pray['prayer_ratio'] = df_pray['Yes_Pray'] / (df_pray['Yes_Pray'] + df_pray['No_Pray'])

In [ ]:
df_well_raw = df_well_raw.loc[df_well_raw["Age"] != "(18, 65)"]

In [ ]:
df_well_raw = df_well_raw.merge(df_pray, on = 'Location', how = 'left')

In [ ]:
def prayer_type(row):
    if row['Religion']=='muslim' and row['Day']=='Tue':
        return "Individual"
    if (row['Religion']=='muslim' and row['Day']=='Fri') or \
       (row['Religion']=='christian' and row['Day']=='Sun'):
        return "Group"
    return "Control"

df_well_raw['Prayer_Type'] = df_well_raw.apply(prayer_type, axis=1)
df_well_raw['Prayer_Type'] = df_well_raw['Prayer_Type'].astype('category')

### DiD on lowest level

In [ ]:
df_well = df_well_raw.query("Reach_Well > 0").copy()

df_well = (df_well_raw
           .groupby(['Location', 'Religion', 'Prayer_Type', 'Gender', 'Age', 'Time', 'Day'], as_index = False)
           [['Total_Poll', 'Reach_Pray', 'Reach_Well', 'Yes_Well', 'No_Well', 'Yes_Pray', 'No_Pray']]
           .sum()
          )


In [ ]:
df_well['pray_ratio'] = df_well['Yes_Pray'] / (df_well['Yes_Pray'] + df_well['No_Pray'])
df_well['well_response'] = (df_well['Yes_Well'] + df_well['No_Well']) / df_well['Reach_Well'] * 100
df_well['after'] = df_well["Time"].map({"After": True, "Before": False})

In [ ]:
df_well.Total_Poll.sum()

Get controls in place

In [ ]:
control_well = (control_vars
                    .loc[control_vars.Day.notna(), 
                    ['ID'] + 
                    weather_controls + 
                    pop_controls + 
                    timing_controls +
                    eco_controls# + 
#                   internet_controls
                    ])

In [ ]:
df_well_glm = (df_well[['ID', 'Location', 'Yes_Well','No_Well','after','Prayer_Type', 'Religion', 'Gender', 'Age', 'well_response', 'pray_ratio']]
                   .merge(control_well, on = 'ID', how = 'left')
                   .drop(columns = 'ID')
                   .dropna()
              )

# Persist for downstream analyses (wild_bootstrap.ipynb, specification_curve.ipynb)
df_well_glm.to_parquet('../midsave/df_well_glm.parquet')
print('df_well_glm rows:', len(df_well_glm), '| locations:', df_well_glm['Location'].nunique())


# A/B Analysis

In [ ]:
df = df_video.copy()

In [ ]:
df_full = df.copy()  # Keep full for descriptives
df = df.dropna(subset=['response_rate']).copy()

# Create treatment indicators
df['is_muslim'] = (df['Religion'] == 'muslim').astype(int)
df['islam_video'] = (df['Video_Style'] == 'Islam').astype(int)

# "Match" = video matches the location's dominant religion
# Islam video + Muslim location = match
# Christian video + Christian location = match
df['match'] = ((df['islam_video'] == 1) & (df['is_muslim'] == 1) |
               (df['islam_video'] == 0) & (df['is_muslim'] == 0)).astype(int)

### Descriptive Statistics

In [ ]:
print(f"\nTotal ads deployed: {len(df_full)}")
print(f"Ads with non-zero reach: {len(df)} (3 dropped)")
print(f"Complete location pairs: {df.groupby('Location').filter(lambda x: len(x)==2)['Location'].nunique()}")
print(f"Locations: {df['Location'].nunique()}")

# Response rate by Religion × Video Style
for rel in ['christian', 'muslim']:
    for vs in ['Christian', 'Islam']:
        s = df[(df['Religion']==rel) & (df['Video_Style']==vs)]
        if len(s) > 0:
            print(f"  {rel:10s} × {vs:10s}: mean={s['response_rate'].mean():.3f}, "
                  f"sd={s['response_rate'].std():.3f}, n={len(s)}")

# Response rate by Match
for m in [0, 1]:
    s = df[df['match'] == m]
    print(f"  Match={m}: mean={s['response_rate'].mean():.3f}, "
          f"sd={s['response_rate'].std():.3f}, n={len(s)}")

# Prayer ratio by Religion × Video Style
df_pr = df.dropna(subset=['prayer_ratio'])
for rel in ['christian', 'muslim']:
    for vs in ['Christian', 'Islam']:
        s = df_pr[(df_pr['Religion']==rel) & (df_pr['Video_Style']==vs)]
        if len(s) > 0:
            print(f"  {rel:10s} × {vs:10s}: mean={s['prayer_ratio'].mean():.3f}, "
                  f"sd={s['prayer_ratio'].std():.3f}, n={len(s)}")

### A/B TEST ON RESPONSE RATE (Engagement)

In [ ]:
# Paired tests (within-location)
pairs = []
for loc in df['Location'].unique():
    sub = df[df['Location'] == loc]
    if len(sub) == 2:
        chr_val = sub[sub['Video_Style']=='Christian']['response_rate'].values[0]
        isl_val = sub[sub['Video_Style']=='Islam']['response_rate'].values[0]
        rel = sub['Religion'].values[0]
        pairs.append({'Location': loc, 'Christian': chr_val, 'Islam': isl_val,
                     'diff': isl_val - chr_val, 'Religion': rel})

pairs_df = pd.DataFrame(pairs)
t_stat, p_val = stats.ttest_rel(pairs_df['Islam'], pairs_df['Christian'])
print(f"  Paired t-test (Islam - Christian): t={t_stat:.3f}, p={p_val:.4f}")
print(f"  Mean difference (Islam - Christian): {pairs_df['diff'].mean():.4f}")
print(f"  N pairs: {len(pairs_df)}")

In [ ]:
# By religion
for rel in ['christian', 'muslim']:
    sub = pairs_df[pairs_df['Religion'] == rel]
    if len(sub) >= 2:
        t, p = stats.ttest_rel(sub['Islam'], sub['Christian'])
        print(f"  {rel:10s}: mean diff={sub['diff'].mean():.4f}, t={t:.3f}, p={p:.4f}, n={len(sub)}")

In [ ]:
# Regression models

def fit_ols(formula, data, label=""):
    m = smf.ols(formula=formula, data=data).fit(cov_type='HC1')
    print(f"\n  {label}")
    print(f"  N={int(m.nobs)}, R²={m.rsquared:.3f}")
    for v in m.params.index:
        if 'C(Location)' not in str(v) and 'Intercept' not in str(v):
            p = m.pvalues[v]
            sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
            print(f"    {str(v):<35s} {m.params[v]:8.4f} ({m.bse[v]:.4f}) p={p:.4f} {sig}")
    return m

In [ ]:
# A1: Simple effect of video style
a1 = fit_ols('response_rate ~ islam_video', df, "A1: Video style only")

# A2: + Religion
a2 = fit_ols('response_rate ~ islam_video * is_muslim', df, "A2: Video style × Religion")

# A3: + Location FE (within-location comparison)
a3 = fit_ols('response_rate ~ islam_video + C(Location)', df, "A3: Video style + Location FE")

# A4: Interaction + Location FE
# Note: is_muslim is absorbed by Location FE
a4 = fit_ols('response_rate ~ islam_video * is_muslim + C(Location)', df,
             "A4: Video style × Religion + Location FE")

# A5: Using "match" indicator + Location FE
a5 = fit_ols('response_rate ~ match + C(Location)', df, "A5: Match + Location FE")

# A6: Match only (no FE, for interpretability)
a6 = fit_ols('response_rate ~ match', df, "A6: Match only")

# A7: Match × Religion (does matching help more for one religion?)
a7 = fit_ols('response_rate ~ match * is_muslim', df, "A7: Match × Religion")

### A/B TEST ON PRAYER RATIO (Content bias)")

In [ ]:
df_pr = df.dropna(subset=['prayer_ratio']).copy()

# Paired test
pairs_pr = []
for loc in df_pr['Location'].unique():
    sub = df_pr[df_pr['Location'] == loc]
    if len(sub) == 2:
        chr_val = sub[sub['Video_Style']=='Christian']['prayer_ratio'].values[0]
        isl_val = sub[sub['Video_Style']=='Islam']['prayer_ratio'].values[0]
        rel = sub['Religion'].values[0]
        pairs_pr.append({'Location': loc, 'Christian': chr_val, 'Islam': isl_val,
                        'diff': isl_val - chr_val, 'Religion': rel})

In [ ]:
pairs_pr_df = pd.DataFrame(pairs_pr)
t_stat_pr, p_val_pr = stats.ttest_rel(pairs_pr_df['Islam'], pairs_pr_df['Christian'])
print(f"  Paired t-test (Islam - Christian): t={t_stat_pr:.3f}, p={p_val_pr:.4f}")
print(f"  Mean difference: {pairs_pr_df['diff'].mean():.4f}")

In [ ]:
# Regression
b1 = fit_ols('prayer_ratio ~ islam_video', df_pr, "B1: Video style → prayer ratio")
b2 = fit_ols('prayer_ratio ~ islam_video * is_muslim', df_pr, "B2: Video style × Religion → prayer ratio")
b3 = fit_ols('prayer_ratio ~ match', df_pr, "B3: Match → prayer ratio")
b4 = fit_ols('prayer_ratio ~ match + C(Location)', df_pr, "B4: Match + Location FE → prayer ratio")

### FIGURES

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 11, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9})

Figure 1: Response rate by Religion × Video Style

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.75, 3.25))

# Panel A: Boxplot by Religion × Video Style
ax = axes[0]
positions = {'christian_Christian': 0, 'christian_Islam': 1,
             'muslim_Christian': 2.5, 'muslim_Islam': 3.5}
colors_box = {'Christian': '#9238E0', 'Islam': '#E09238'}

for rel in ['christian', 'muslim']:
    for vs in ['Christian', 'Islam']:
        s = df[(df['Religion']==rel) & (df['Video_Style']==vs)]
        if len(s) > 0:
            key = f'{rel}_{vs}'
            bp = ax.boxplot([s['response_rate'].values], positions=[positions[key]],
                           widths=0.7, patch_artist=True, showfliers=True)
            bp['boxes'][0].set_facecolor(colors_box[vs])
            bp['boxes'][0].set_alpha(1)
            bp['boxes'][0].set_edgecolor('black')
            bp['medians'][0].set_color('black')

ax.set_xticks([0.5, 3.0])
ax.set_xticklabels(['Christian', 'Muslim'])
ax.set_ylabel('Response rate (%)')
ax.set_title('A. Response rate by\nReligion × Video Style', fontweight='bold')
ax.legend(handles=[Patch(fc='#9238E0', ec='k', alpha=1, label='Heaven'),
                   Patch(fc='#E09238', ec='k', alpha=1, label='Jannah')],
          loc='upper left')

# Panel B: Paired differences by location
ax = axes[1]
pairs_chr = pairs_df[pairs_df['Religion'] == 'christian'].sort_values('diff')
pairs_mus = pairs_df[pairs_df['Religion'] == 'muslim'].sort_values('diff')

y_pos = 0
labels = []
for _, row in pairs_chr.iterrows():
    color = '#9238E0'
    ax.barh(y_pos, row['diff'], color=color, alpha=1, edgecolor='black', linewidth=0.5, height=0.7)
    labels.append(row['Location'])
    y_pos += 1

y_pos += 0.5  # gap between religions
for _, row in pairs_mus.iterrows():
    color = '#E09238'
    ax.barh(y_pos, row['diff'], color=color, alpha=1, edgecolor='black', linewidth=0.5, height=0.7)
    labels.append(row['Location'])
    y_pos += 1

ax.set_yticks(list(range(len(pairs_chr))) + [i + len(pairs_chr) + 0.5 for i in range(len(pairs_mus))])
ax.set_yticklabels(labels)
ax.axvline(0, color='grey', ls='--', alpha=0.5)
ax.set_xlabel('Islam − Christian video\n(response rate diff., pp)')
ax.set_title('B. Within-location paired\ndifferences', fontweight='bold')
ax.legend(handles=[Patch(fc='#9238E0', ec='k', alpha=1, label='Christian loc.'),
                   Patch(fc='#E09238', ec='k', alpha=1, label='Muslim loc.')],
          loc='lower right')

# Panel C: Match vs No-Match
ax = axes[2]
for m, color, xpos, label in [(0, '#bdbdbd', 0, 'No match'),
                                (1, '#38E092', 1, 'Match')]:
    s = df[df['match'] == m]
    mean = s['response_rate'].mean()
    se = s['response_rate'].std() / np.sqrt(len(s))
    ax.bar(xpos, mean, width=0.6, color=color, edgecolor='black', linewidth=0.5)
    ax.errorbar(xpos, mean, yerr=1.96*se, fmt='none', color='black', capsize=4)
    ax.text(xpos, mean + 1.96*se + 0.05, f'n={len(s)}', ha='center')

ax.set_xticks([0, 1])
ax.set_xticklabels(['No match', 'Match'])
ax.set_ylabel('Response rate (%)')
ax.set_title('C. Religious identification\neffect', fontweight='bold')
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

plt.tight_layout()
plt.savefig('../output/fig_ab_test.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig_ab_test.pdf', bbox_inches='tight')
plt.close()

Figure 2: Prayer ratio analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Panel A: Prayer ratio by Religion × Video Style
ax = axes[0]
for rel, xoff in [('christian', 0), ('muslim', 2.5)]:
    for vs, color, dx in [('Christian', '#9238E0', 0), ('Islam', '#E09238', 1)]:
        s = df_pr[(df_pr['Religion']==rel) & (df_pr['Video_Style']==vs)]
        if len(s) > 0:
            mean = s['prayer_ratio'].mean()
            se = s['prayer_ratio'].std() / np.sqrt(len(s))
            ax.bar(xoff+dx, mean, width=0.8, color=color, edgecolor='black',
                   linewidth=0.5, alpha=1)
            ax.errorbar(xoff+dx, mean, yerr=1.96*se, fmt='none', color='black', capsize=3)
            ax.text(xoff+dx, mean+1.96*se+0.005, f'n={len(s)}', ha='center', fontsize=8)

ax.set_xticks([0.5, 3.0])
ax.set_xticklabels(['Christian loc.', 'Muslim loc.'], fontsize=10)
ax.set_ylabel('Prayer ratio (prop. Yes)', fontsize=10)
ax.set_title('A. Prayer ratio by\nReligion × Video Style', fontsize=11, fontweight='bold')
ax.legend(handles=[Patch(fc='#9238E0', ec='k', alpha=1, label='Heaven'),
                   Patch(fc='#E09238', ec='k', alpha=1, label='Jannah')],
          fontsize=8)

# Panel B: Paired differences in prayer ratio
ax = axes[1]
pr_pairs = pairs_pr_df.sort_values('diff')
colors_pr = ['#9238E0' if r=='christian' else '#E09238' for r in pr_pairs['Religion']]
ax.barh(range(len(pr_pairs)), pr_pairs['diff'], color=colors_pr, alpha=1,
        edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(pr_pairs)))
ax.set_yticklabels(pr_pairs['Location'], fontsize=8)
ax.axvline(0, color='grey', ls='--', alpha=0.5)
ax.set_xlabel('Islam − Christian video\n(prayer ratio diff.)', fontsize=9)
ax.set_title('B. Within-location differences\nin prayer ratio', fontsize=11, fontweight='bold')
ax.legend(handles=[Patch(fc='#9238E0', ec='k', alpha=1, label='Christian loc.'),
                   Patch(fc='#E09238', ec='k', alpha=1, label='Muslim loc.')],
          loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('../output/fig_ab_prayer_ratio.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig_ab_prayer_ratio.pdf', bbox_inches='tight')
print("  → fig_ab_prayer_ratio saved")
plt.close()

### LATEX TABLES

In [ ]:
def sig_stars(p):
    if p < 0.001: return '^{***}'
    if p < 0.01: return '^{**}'
    if p < 0.05: return '^{*}'
    return ''

def fmt(model, var):
    if model is None or var not in model.params.index:
        return '', ''
    c = model.params[var]; se = model.bse[var]; p = model.pvalues[var]
    if se > 100: return '', ''
    return f'{c:.4f}{sig_stars(p)}', f'({se:.4f})'

Table: A/B Test on Response Rate

In [ ]:
lines = []
lines.append(r'\begin{table}[!htbp] \centering')
lines.append(r'  \caption{A/B Test --- Effect of Video Style on Response Rate}')
lines.append(r'  \label{tab:ab_response_rate}')
lines.append(r'\begin{tabular}{@{\extracolsep{5pt}}lccccc}')
lines.append(r'\\[-1.8ex]\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r'& \multicolumn{5}{c}{\textit{Dependent variable: Response rate (\%)}} \\')
lines.append(r'\cline{2-6}')
lines.append(r'\\[-1.8ex]')
lines.append(r'& (1) & (2) & (3) & (4) & (5) \\')
lines.append(r'\hline \\[-1.8ex]')

vars_to_show = [
    ('islam_video', 'Islam video (Jannah)'),
    ('is_muslim', 'Muslim location'),
    ('islam_video:is_muslim', 'Islam video $\\times$ Muslim loc.'),
    ('match', 'Match (same religion)'),
]

for var, label in vars_to_show:
    coefs = [fmt(m, var) for m in [a1, a2, a3, a4, a6]]
    c_line = f' {label} & ' + ' & '.join([f'${c}$' if c else '' for c,s in coefs]) + r' \\'
    s_line = ' & ' + ' & '.join([f'${s}$' if s else '' for c,s in coefs]) + r' \\'
    lines.append(c_line)
    lines.append(s_line)

lines.append(r'\hline \\[-1.8ex]')
lines.append(r' Location fixed effects & No & No & Yes & Yes & No \\')
lines.append(r' Observations & ' + ' & '.join([str(int(m.nobs)) for m in [a1,a2,a3,a4,a6]]) + r' \\')
lines.append(r' $R^2$ & ' + ' & '.join([f'{m.rsquared:.3f}' for m in [a1,a2,a3,a4,a6]]) + r' \\')
lines.append(r'\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r"\multicolumn{6}{l}{\parbox{13cm}{\textit{Note:} $^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. OLS with HC1 robust standard errors. Response rate is defined as total poll responses divided by ad reach, multiplied by 100. Match equals one when the video iconography corresponds to the dominant religion of the ad's location (Heaven for Christian, Jannah for Muslim locations). Columns (3)--(4) include location fixed effects, exploiting the within-location paired design.}} \\")
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

table_rr = '\n'.join(lines)

# ── Table: A/B Test on Prayer Ratio ──
lines2 = []
lines2.append(r'\begin{table}[!htbp] \centering')
lines2.append(r'  \caption{A/B Test --- Effect of Video Style on Reported Prayer}')
lines2.append(r'  \label{tab:ab_prayer_ratio}')
lines2.append(r'\begin{tabular}{@{\extracolsep{5pt}}lcccc}')
lines2.append(r'\\[-1.8ex]\hline')
lines2.append(r'\hline \\[-1.8ex]')
lines2.append(r'& \multicolumn{4}{c}{\textit{Dependent variable: Prayer ratio (prop.\ Yes)}} \\')
lines2.append(r'\cline{2-5}')
lines2.append(r'\\[-1.8ex]')
lines2.append(r'& (1) & (2) & (3) & (4) \\')
lines2.append(r'\hline \\[-1.8ex]')

vars_b = [
    ('islam_video', 'Islam video (Jannah)'),
    ('is_muslim', 'Muslim location'),
    ('islam_video:is_muslim', 'Islam video $\\times$ Muslim loc.'),
    ('match', 'Match (same religion)'),
]

for var, label in vars_b:
    coefs = [fmt(m, var) for m in [b1, b2, b3, b4]]
    c_line = f' {label} & ' + ' & '.join([f'${c}$' if c else '' for c,s in coefs]) + r' \\'
    s_line = ' & ' + ' & '.join([f'${s}$' if s else '' for c,s in coefs]) + r' \\'
    lines2.append(c_line)
    lines2.append(s_line)

lines2.append(r'\hline \\[-1.8ex]')
lines2.append(r' Location fixed effects & No & No & No & Yes \\')
lines2.append(r' Observations & ' + ' & '.join([str(int(m.nobs)) for m in [b1,b2,b3,b4]]) + r' \\')
lines2.append(r' $R^2$ & ' + ' & '.join([f'{m.rsquared:.3f}' for m in [b1,b2,b3,b4]]) + r' \\')
lines2.append(r'\hline')
lines2.append(r'\hline \\[-1.8ex]')
lines2.append(r"\multicolumn{5}{l}{\parbox{11cm}{\textit{Note:} $^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. OLS with HC1 robust standard errors. Prayer ratio is the share of respondents answering Yes to the question ``Do you pray daily?'' This table tests whether the video's religious iconography biases the substantive response, independently of its effect on engagement (Table~\ref{tab:ab_response_rate}).}} \\")
lines2.append(r'\end{tabular}')
lines2.append(r'\end{table}')

table_pr = '\n'.join(lines2)

In [ ]:
# ── Descriptive table ──
lines3 = []
lines3.append(r'\begin{table}[!htbp] \centering')
lines3.append(r'  \caption{Descriptive Statistics --- Prayer Video Poll Ads (A/B Test)}')
lines3.append(r'  \label{tab:ab_descriptive}')
lines3.append(r'\begin{tabular}{@{\extracolsep{5pt}}llcrrr}')
lines3.append(r'\\[-1.8ex]\hline')
lines3.append(r'\hline \\[-1.8ex]')
lines3.append(r'Location & Religion & Video Style & Resp.\ Rate (\%) & Prayer Ratio & Match \\')
lines3.append(r'\hline \\[-1.8ex]')

for _, row in df.sort_values(['Religion', 'Location', 'Video_Style']).iterrows():
    match_str = 'Yes' if row['match'] == 1 else 'No'
    rr = f"{row['response_rate']:.3f}" if pd.notna(row['response_rate']) else '---'
    pr = f"{row['prayer_ratio']:.3f}" if pd.notna(row['prayer_ratio']) else '---'
    loc = row['Location'].replace('ï', r'\"{\i}').replace('â', r'\^{a}').replace('é', r'\'{e}')
    lines3.append(f" {loc} & {row['Religion']} & {row['Video_Style']} & {rr} & {pr} & {match_str} \\\\")

lines3.append(r'\hline')
lines3.append(r'\hline \\[-1.8ex]')
lines3.append(r"\multicolumn{6}{l}{\parbox{12cm}{\textit{Note:} Each location received both video styles (Heaven/Christian and Jannah/Islam). Match indicates whether the video's iconography corresponds to the dominant religion of the location. Three ads with zero reach are excluded.}} \\")
lines3.append(r'\end{tabular}')
lines3.append(r'\end{table}')

table_desc = '\n'.join(lines3)


# Write all tables
full_tex = f"""% =============================================================
% A/B TEST TABLES --- Auto-generated
% =============================================================

{table_rr}

\\clearpage

{table_pr}

\\clearpage

{table_desc}
"""

with open('../output/tables_ab.tex', 'w') as f:
    f.write(full_tex)

In [ ]:
table_rr

### KEY FINDINGS

In [ ]:
match_coef = a6.params.get('match', np.nan)
match_p = a6.pvalues.get('match', np.nan)
match_fe_coef = a5.params.get('match', np.nan)
match_fe_p = a5.pvalues.get('match', np.nan)

print(f"""
RESPONSE RATE (engagement):

  Overall Islam vs Christian video (paired t-test):
    Mean diff = {pairs_df['diff'].mean():.4f} pp, p = {p_val:.4f}
    {'Significant' if p_val < 0.05 else 'Not significant'} overall effect

  Match effect (same-religion iconography):
    Without FE: {match_coef:+.4f} pp (p = {match_p:.4f})
    With loc FE: {match_fe_coef:+.4f} pp (p = {match_fe_p:.4f})
    {'Matching iconography significantly boosts' if match_p < 0.05 else 'No significant effect of matching iconography on'} response rates

  By religion:
    Christian locations: mean diff = {pairs_df[pairs_df['Religion']=='christian']['diff'].mean():.4f}
    Muslim locations:    mean diff = {pairs_df[pairs_df['Religion']=='muslim']['diff'].mean():.4f}

PRAYER RATIO (content bias):

  Paired t-test: mean diff = {pairs_pr_df['diff'].mean():.4f}, p = {p_val_pr:.4f}
  {'Significant' if p_val_pr < 0.05 else 'No significant'} effect of video style on reported prayer

  This means the video iconography does {'bias' if p_val_pr < 0.05 else 'NOT significantly bias'} the substantive response to 
  "Do you pray daily?"
""")


# Creating the Latex Tables

### Prepare data

In [ ]:
df = df_well_glm.copy()
df['total_responses'] = df['Yes_Well'] + df['No_Well']
df['prop_good'] = df['Yes_Well'] / df['total_responses']
df['after_int'] = df['after'].astype(int)
df['is_muslim'] = (df['Religion'] == 'muslim').astype(int)
df['is_female'] = (df['Gender'] == 'Female').astype(int)
df['age_25_49'] = (df['Age'] == '(25, 49)').astype(int)
df['age_50_65'] = (df['Age'] == '(50, 65)').astype(int)

df['own_prayer_day'] = 0
df.loc[(df['Religion']=='muslim') & (df['day_of_week_start']=='Friday'), 'own_prayer_day'] = 1
df.loc[(df['Religion']=='christian') & (df['day_of_week_start']=='Sunday'), 'own_prayer_day'] = 1

weather_covs = ['temperature_2m','precipitation','cloud_cover',
                'wind_speed_10m','relative_humidity_2m']
weather_str = ' + '.join(weather_covs)
demog_str = 'is_female + age_25_49 + age_50_65'

def get_nonconstant(data, covs):
    return [w for w in covs if data[w].nunique() > 1]

### Fit Models

In [ ]:
def fit_glm(formula, data, cluster_var='Location', use_hc=False):
    kw = {}
    if cluster_var and not use_hc:
        kw = dict(cov_type='cluster', cov_kwds={'groups': data[cluster_var]})
    elif use_hc:
        kw = dict(cov_type='HC1')
    m = smf.glm(formula=formula, data=data,
                family=sm.families.Binomial()).fit(**kw)
    return m

In [ ]:
def fit_wls(formula, data, cluster_var='Location'):
    m = smf.wls(formula=formula, data=data,
                weights=data['total_responses']
                ).fit(cov_type='cluster', cov_kwds={'groups': data[cluster_var]})
    return m

Communal Prayer DID

In [ ]:
df_comm = df[df['day_of_week_start'].isin(['Friday','Sunday'])].copy()

cm1 = fit_glm('Yes_Well + No_Well ~ after_int * own_prayer_day', df_comm)
cm2 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str}', df_comm)
cm3 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {weather_str}', df_comm)
cm4 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {weather_str} + C(Location)', df_comm)

Individual Prayer DID

In [ ]:
df_tue = df[df['day_of_week_start']=='Tuesday'].copy()
w_tue = get_nonconstant(df_tue, weather_covs)
w_tue_str = ' + '.join(w_tue)

im1 = fit_glm('Yes_Well + No_Well ~ after_int * is_muslim', df_tue)
im2 = fit_glm(f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str}', df_tue)
im3 = fit_glm(f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str} + {w_tue_str}', df_tue)
im4 = fit_glm(f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str} + {w_tue_str} + C(Location)', df_tue)

### Robustness Checks

More than 3 responses

In [ ]:
df_comm_big = df_comm[df_comm['total_responses'] > 3].copy()
rm1 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {weather_str} + C(Location)', df_comm_big)

No Ain Sebaa

In [ ]:
df_comm_noain = df_comm[df_comm['Location'] != 'Aïn Sebaâ'].copy()
rm2 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {weather_str} + C(Location)', df_comm_noain)

Weighted Least Squares

In [ ]:
rm3 = fit_wls(f'prop_good ~ after_int * own_prayer_day + {demog_str} + {weather_str} + C(Location)', df_comm)

Muslim Only

In [ ]:
df_comm_mus = df_comm[df_comm['Religion']=='muslim'].copy()
w_mus = get_nonconstant(df_comm_mus, weather_covs)
rm4 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {" + ".join(w_mus)} + C(Location)', df_comm_mus)

Christian Only

In [ ]:
df_comm_chr = df_comm[df_comm['Religion']=='christian'].copy()
w_chr = get_nonconstant(df_comm_chr, weather_covs)
rm5 = fit_glm(f'Yes_Well + No_Well ~ after_int * own_prayer_day + {demog_str} + {" + ".join(w_chr)} + C(Location)', df_comm_chr)

Falsification I -- Sunday and Muslims

In [ ]:
df_f1 = df[(df['day_of_week_start']=='Sunday')&(df['Religion']=='muslim')].copy()
w_f1 = get_nonconstant(df_f1, weather_covs)
fm1 = fit_glm(f'Yes_Well + No_Well ~ after_int + {demog_str} + {" + ".join(w_f1)}', df_f1)

Falsification II -- Friday and Christian

In [ ]:
df_f2 = df[(df['day_of_week_start']=='Friday')&(df['Religion']=='christian')].copy()
w_f2 = get_nonconstant(df_f2, weather_covs)
fm2 = fit_glm(f'Yes_Well + No_Well ~ after_int + {demog_str} + {" + ".join(w_f2)}', df_f2)

### Stargazer-like formatting
Stargazer does not work with GLM

In [ ]:
def sig_stars(p):
    if p < 0.001: return '^{***}'
    if p < 0.01: return '^{**}'
    if p < 0.05: return '^{*}'
    return ''

def fmt_coef(model, var):
    """Return (coef_str, se_str) or ('', '') if var not in model."""
    if model is None or var not in model.params.index:
        return '', ''
    c = model.params[var]
    se = model.bse[var]
    p = model.pvalues[var]
    stars = sig_stars(p)
    # Handle blown-up SEs (absorbed coefficients)
    if se > 100:
        return '', ''
    return f'{c:.4f}{stars}', f'({se:.4f})'

def make_stargazer_table(
    models,            # list of fitted models
    var_order,         # list of variable names to display
    var_labels,        # dict: var_name -> display label
    col_labels,        # list of column header labels
    dep_var_label,     # dependent variable label
    title,             # table title
    label,             # LaTeX label
    note='',           # table note
    extra_rows=None,   # list of (label, val1, val2, ...) tuples
    n_obs=None,        # list of N observations (override)
    n_resp=None,       # list of N responses
    n_clust=None,      # list of N clusters
    model_type='Binomial GLM'
):
    """Generate a stargazer-style LaTeX table."""
    ncols = len(models)
    
    lines = []
    lines.append(r'\begin{table}[!htbp] \centering')
    lines.append(f'  \\caption{{{title}}}')
    lines.append(f'  \\label{{{label}}}')
    col_spec = 'l' + 'c' * ncols
    lines.append(f'\\begin{{tabular}}{{@{{\\extracolsep{{5pt}}}}{col_spec}}}')
    lines.append(r'\\[-1.8ex]\hline')
    lines.append(r'\hline \\[-1.8ex]')
    
    # Dependent variable header
    lines.append(f'& \\multicolumn{{{ncols}}}{{c}}{{\\textit{{Dependent variable: {dep_var_label}}}}} \\\\')
    lines.append(f'\\cline{{2-{ncols+1}}}')
    lines.append(r'\\[-1.8ex]')
    
    # Column labels
    header = ' & '.join([f'({i+1})' for i in range(ncols)])
    lines.append(f'& {header} \\\\')
    if col_labels:
        lbl_line = ' & '.join(col_labels)
        lines.append(f'& {lbl_line} \\\\')
    lines.append(r'\hline \\[-1.8ex]')
    
    # Coefficients
    for var in var_order:
        label_str = var_labels.get(var, var.replace('_', r'\_'))
        coefs = [fmt_coef(m, var) for m in models]
        
        # Coefficient row
        coef_cells = []
        for c, s in coefs:
            if c:
                coef_cells.append(f'${c}$')
            else:
                coef_cells.append('')
        line = f' {label_str} & ' + ' & '.join(coef_cells) + ' \\\\'
        lines.append(line)
        
        # SE row
        se_cells = []
        for c, s in coefs:
            if s:
                se_cells.append(f'${s}$')
            else:
                se_cells.append('')
        line = ' & ' + ' & '.join(se_cells) + ' \\\\'
        lines.append(line)
    
    lines.append(r'\hline \\[-1.8ex]')
    
    # Extra rows (controls indicators etc.)
    if extra_rows:
        for row in extra_rows:
            label_str = row[0]
            vals = row[1:]
            line = f' {label_str} & ' + ' & '.join(vals) + ' \\\\'
            lines.append(line)
    
    # N observations
    if n_obs:
        line = ' Observations (ads) & ' + ' & '.join([str(n) for n in n_obs]) + ' \\\\'
        lines.append(line)
    else:
        obs = [str(int(m.nobs)) if m else '' for m in models]
        line = ' Observations (ads) & ' + ' & '.join(obs) + ' \\\\'
        lines.append(line)
    
    if n_resp:
        line = ' Total responses & ' + ' & '.join([f'{n:,}' for n in n_resp]) + ' \\\\'
        lines.append(line)
    
    if n_clust:
        line = ' Clusters & ' + ' & '.join([str(n) for n in n_clust]) + ' \\\\'
        lines.append(line)
    
    lines.append(r'\hline')
    lines.append(r'\hline \\[-1.8ex]')
    
    # Note
    if note:
        lines.append(f'\\multicolumn{{{ncols+1}}}{{l}}{{\\parbox{{{ncols*2.2+2}cm}}{{\\textit{{Note:}} {note}}}}} \\\\')
    else:
        lines.append(f'\\multicolumn{{{ncols+1}}}{{l}}{{\\textit{{Note:}} $^{{*}}$p$<$0.05; $^{{**}}$p$<$0.01; $^{{***}}$p$<$0.001}} \\\\')
    
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    
    return '\n'.join(lines)

Communal Prayer

In [ ]:
comm_resp = df_comm['total_responses'].sum()
comm_clust = df_comm['Location'].nunique()

table1 = make_stargazer_table(
    models=[cm1, cm2, cm3, cm4],
    var_order=[
        'after_int',
        'own_prayer_day',
        'after_int:own_prayer_day',
        'is_female',
        'age_25_49',
        'age_50_65',
        'temperature_2m',
        'precipitation',
        'cloud_cover',
        'wind_speed_10m',
        'relative_humidity_2m',
    ],
    var_labels={
        'after_int': 'After',
        'own_prayer_day': 'Own prayer day',
        'after_int:own_prayer_day': 'After $\\times$ Own prayer day',
        'is_female': 'Female',
        'age_25_49': 'Age 25--49',
        'age_50_65': 'Age 50--65',
        'temperature_2m': 'Temperature (\\textdegree C)',
        'precipitation': 'Precipitation (mm)',
        'cloud_cover': 'Cloud cover (\\%)',
        'wind_speed_10m': 'Wind speed (m/s)',
        'relative_humidity_2m': 'Relative humidity (\\%)',
    },
    col_labels=['', '+ Demog.', '+ Weather', '+ Location FE'],
    dep_var_label='Pr(Good)',
    title='Communal Prayer DID --- Instagram Video Poll Ads (Friday + Sunday)',
    label='tab:communal_did',
    extra_rows=[
        ('Demographics', 'No', 'Yes', 'Yes', 'Yes'),
        ('Weather controls', 'No', 'No', 'Yes', 'Yes'),
        ('Location fixed effects', 'No', 'No', 'No', 'Yes'),
    ],
    n_obs=[250, 250, 250, 250],
    n_resp=[comm_resp]*4,
    n_clust=[comm_clust]*4,
    note='$^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. Binomial GLM (logit link). Standard errors clustered at the location level in parentheses. Each observation is an ad; the dependent variable is the number of Good responses out of total responses. Column (4) is the preferred specification.'
)

Individual Prayer

In [ ]:
tue_resp = df_tue['total_responses'].sum()
tue_clust = df_tue['Location'].nunique()

table2 = make_stargazer_table(
    models=[im1, im2, im3, im4],
    var_order=[
        'after_int',
        'is_muslim',
        'after_int:is_muslim',
        'is_female',
        'age_25_49',
        'age_50_65',
        'temperature_2m',
        'precipitation',
        'cloud_cover',
        'wind_speed_10m',
        'relative_humidity_2m',
    ],
    var_labels={
        'after_int': 'After',
        'is_muslim': 'Muslim location',
        'after_int:is_muslim': 'After $\\times$ Muslim location',
        'is_female': 'Female',
        'age_25_49': 'Age 25--49',
        'age_50_65': 'Age 50--65',
        'temperature_2m': 'Temperature (\\textdegree C)',
        'precipitation': 'Precipitation (mm)',
        'cloud_cover': 'Cloud cover (\\%)',
        'wind_speed_10m': 'Wind speed (m/s)',
        'relative_humidity_2m': 'Relative humidity (\\%)',
    },
    col_labels=['', '+ Demog.', '+ Weather', '+ Location FE'],
    dep_var_label='Pr(Good)',
    title='Individual Prayer DID --- Instagram Video Poll Ads (Tuesday)',
    label='tab:individual_did',
    extra_rows=[
        ('Demographics', 'No', 'Yes', 'Yes', 'Yes'),
        ('Weather controls', 'No', 'No', 'Yes', 'Yes'),
        ('Location fixed effects', 'No', 'No', 'No', 'Yes'),
    ],
    n_obs=[119, 119, 119, 119],
    n_resp=[tue_resp]*4,
    n_clust=[tue_clust]*4,
    note='$^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. Binomial GLM (logit link). Standard errors clustered at the location level in parentheses. In column (4), the Muslim location main effect is absorbed by location fixed effects (each location is either Muslim- or Christian-majority). The interaction remains identified through within-location before--after variation.'
)



Robustness Checks

In [ ]:
table3 = make_stargazer_table(
    models=[cm4, rm1, rm2, rm3, rm4, rm5],
    var_order=[
        'after_int',
        'own_prayer_day',
        'after_int:own_prayer_day',
        'is_female',
        'age_25_49',
        'age_50_65',
    ],
    var_labels={
        'after_int': 'After',
        'own_prayer_day': 'Own prayer day',
        'after_int:own_prayer_day': 'After $\\times$ Own prayer day',
        'is_female': 'Female',
        'age_25_49': 'Age 25--49',
        'age_50_65': 'Age 50--65',
    },
    col_labels=[
        '\\scriptsize Preferred',
        '\\scriptsize Drop small',
        '\\scriptsize No A\\"{\\i}n S.',
        '\\scriptsize LPM',
        '\\scriptsize Muslim',
        '\\scriptsize Christian'
    ],
    dep_var_label='Pr(Good)',
    title='Robustness Checks --- Communal Prayer DID',
    label='tab:robustness',
    extra_rows=[
        ('Demographics', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'),
        ('Weather controls', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'),
        ('Location fixed effects', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'),
        ('Sample restriction', 'None', '$>$3 resp.', 'Excl. A\\"{\\i}n S.', 'None', 'Muslim only', 'Christian only'),
        ('Estimator', 'Logit', 'Logit', 'Logit', 'WLS', 'Logit', 'Logit'),
    ],
    n_obs=[250, len(df_comm_big), len(df_comm_noain), 250, len(df_comm_mus), len(df_comm_chr)],
    n_resp=[comm_resp, df_comm_big['total_responses'].sum(),
            df_comm_noain['total_responses'].sum(), comm_resp,
            df_comm_mus['total_responses'].sum(), df_comm_chr['total_responses'].sum()],
    n_clust=[14, df_comm_big['Location'].nunique(), 13, 14, 6, 8],
    note='$^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. Column (1) reproduces the preferred specification from Table~\\ref{tab:communal_did}. Column (2) drops ads with $\\leq$3 responses. Column (3) excludes A\\"{\\i}n Seba\\^{a}. Column (4) estimates a linear probability model by weighted OLS. Columns (5)--(6) estimate separately by religion. Standard errors clustered at the location level.'
)

Falsification Tests

In [ ]:
table4 = make_stargazer_table(
    models=[fm1, fm2],
    var_order=[
        'after_int',
        'is_female',
        'age_25_49',
        'age_50_65',
        'temperature_2m',
        'precipitation',
        'cloud_cover',
        'wind_speed_10m',
        'relative_humidity_2m',
    ],
    var_labels={
        'after_int': 'After',
        'is_female': 'Female',
        'age_25_49': 'Age 25--49',
        'age_50_65': 'Age 50--65',
        'temperature_2m': 'Temperature (\\textdegree C)',
        'precipitation': 'Precipitation (mm)',
        'cloud_cover': 'Cloud cover (\\%)',
        'wind_speed_10m': 'Wind speed (m/s)',
        'relative_humidity_2m': 'Relative humidity (\\%)',
    },
    col_labels=['Sunday, Muslim loc.', 'Friday, Christian loc.'],
    dep_var_label='Pr(Good)',
    title='Falsification Tests --- Before/After on Non-Prayer Days',
    label='tab:falsification',
    extra_rows=[
        ('Demographics', 'Yes', 'Yes'),
        ('Weather controls', 'Yes', 'Yes'),
        ('Location fixed effects', 'No', 'No'),
        ('Prayer event present', 'No', 'No'),
    ],
    n_obs=[len(df_f1), len(df_f2)],
    n_resp=[df_f1['total_responses'].sum(), df_f2['total_responses'].sum()],
    n_clust=[df_f1['Location'].nunique(), df_f2['Location'].nunique()],
    note='$^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. Binomial GLM (logit link). These specifications test for before--after differences on days when no communal prayer event occurs for the dominant religion of the location. Column (1): Sunday ads in Muslim-majority locations (Sunday is not the Muslim prayer day). Column (2): Friday ads in Christian-majority locations (Friday is not the Christian prayer day). A significant After coefficient would suggest that generic time-of-day effects, rather than prayer, drive the DID estimates.'
)

Descriptive Statistics

In [ ]:
desc_lines = []
desc_lines.append(r'\begin{table}[!htbp] \centering')
desc_lines.append(r'  \caption{Descriptive Statistics --- Instagram Video Poll Ads}')
desc_lines.append(r'  \label{tab:descriptive}')
desc_lines.append(r'\begin{tabular}{@{\extracolsep{5pt}}llrrrr}')
desc_lines.append(r'\\[-1.8ex]\hline')
desc_lines.append(r'\hline \\[-1.8ex]')
desc_lines.append(r'Prayer Type & Religion & Day & Ads & Responses & Prop.\ Good \\')
desc_lines.append(r'\hline \\[-1.8ex]')

for pt in ['Control', 'Group', 'Individual']:
    for rel in ['Christian', 'Muslim']:
        for day in ['Tuesday', 'Friday', 'Sunday']:
            s = df[(df['Prayer_Type']==pt) & 
                   (df['Religion']==rel.lower()) & 
                   (df['day_of_week_start']==day)]
            if len(s) == 0:
                continue
            t = s['total_responses'].sum()
            p = s['Yes_Well'].sum() / t if t > 0 else 0
            desc_lines.append(
                f' {pt} & {rel} & {day} & {len(s)} & {t:,} & {p:.3f} \\\\'
            )

desc_lines.append(r'\hline \\[-1.8ex]')

# Totals
total_ads = len(df)
total_resp = df['total_responses'].sum()
total_prop = df['Yes_Well'].sum() / total_resp
desc_lines.append(f' \\textbf{{Total}} & & & \\textbf{{{total_ads}}} & \\textbf{{{total_resp:,}}} & \\textbf{{{total_prop:.3f}}} \\\\')

desc_lines.append(r'\hline')
desc_lines.append(r'\hline \\[-1.8ex]')
desc_lines.append(r'\multicolumn{6}{l}{\parbox{12cm}{\textit{Note:} Prayer Type is a deterministic function of Religion $\times$ Day: Individual = Muslim $\times$ Tuesday (Dhuhr prayer); Group = Muslim $\times$ Friday (Jumuah) or Christian $\times$ Sunday (Church service); Control = all other combinations. Prop.\ Good is the weighted proportion reporting Good across all ads in each cell.}} \\')
desc_lines.append(r'\end{tabular}')
desc_lines.append(r'\end{table}')

table5 = '\n'.join(desc_lines)

In [ ]:
cm4.summary()

### Main DID figure (Figure 1) and heterogeneity (Figure 2)

In [ ]:
# Helpers used by the Figure 1/2 cell (originally defined in the removed analysis-log section)
def wilson_ci(yes, n, z=1.96):
    if n == 0: return np.nan, np.nan, np.nan
    p = yes / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / d
    w = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return p, c - w, c + w

log = print

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 11, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9})

Figure 1: Main DID panels

In [ ]:
def plot_2x2(ax, data_groups, labels, colors, title, xlabels_bottom=None):
    """Plot 2×2 bar chart with Wilson CIs."""
    xpos = list(range(len(data_groups)))
    for i, (yes, n, lbl) in enumerate(data_groups):
        p, lo, hi = wilson_ci(yes, n)
        ax.bar(i, p, width=0.75, color=colors[i], edgecolor='black', linewidth=0.5)
        ax.errorbar(i, p, yerr=[[p-lo],[hi-p]], fmt='none', color='black', capsize=3, linewidth=1.2)
        ax.text(i, hi+0.008, f'n={n:,}', ha='center', color='#333')
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0.38, 0.68)
    ax.axhline(0.5, color='grey', ls='--', alpha=0.4, linewidth=0.8)
    ax.set_ylabel("Prop. 'Good'")
    if xlabels_bottom:
        for pos, txt in xlabels_bottom:
            ax.text(pos, -0.20, txt, ha='center', va='top', transform=ax.get_xaxis_transform())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.5))

# Panel A: Communal DID
groups_a = []
for opd in [0, 1]:
    for aft in [0, 1]:
        s = df_comm[(df_comm['own_prayer_day']==opd)&(df_comm['after_int']==aft)]
        groups_a.append((s['Yes_Well'].sum(), s['total_responses'].sum(), ''))

plot_2x2(axes[0], groups_a,
         ['Before','After','Before','After'],
         ['#C197E8','#9238E0','#E8C197','#E09238'],
         'A. Communal Prayer DID\n(Friday + Sunday)',
         [(0.5, "Other religion's\nprayer day"), (2.5, "Own religion's\nprayer day")])

# Panel B: Individual Prayer DID (Tuesday: Muslim vs Christian)
groups_b = []
df_tue = df[df['day_of_week_start']=='Tuesday']
for rel_label, rel_val in [('christian', 'christian'), ('muslim', 'muslim')]:
    for aft in [0,1]:
        s = df_tue[(df_tue['Religion']==rel_val) & (df_tue['after_int']==aft)]
        groups_b.append((s['Yes_Well'].sum(), s['total_responses'].sum(), ''))

plot_2x2(axes[1], groups_b,
         ['Before','After','Before','After'],
         ['#c7e9c0','#38E092','#97E7E8','#38DAE0'],
         'B. Individual Prayer DID\n(Tuesday: Muslim vs Christian)',
         [(0.5, 'Christian\n(No Dhuhr)'), (2.5, 'Muslim\n(Dhuhr at midday)')])

# Panel C: Falsification (Sun Muslim + Fri Christian — no prayer events)
groups_c = []
for aft in [0,1]:
    s = df_f1[df_f1['after_int']==aft]
    groups_c.append((s['Yes_Well'].sum(), s['total_responses'].sum(), ''))
for aft in [0,1]:
    s = df_f2[df_f2['after_int']==aft]
    groups_c.append((s['Yes_Well'].sum(), s['total_responses'].sum(), ''))

plot_2x2(axes[2], groups_c,
         ['Before','After','Before','After'],
         ['#9897E8','#3E38E0','#DBDBDB','#707070'],
         'C. Falsification Tests\n(No prayer event)',
         [(0.5, 'Sun: Muslim\n(Not their day)'), (2.5, 'Fri: Christian\n(Not their day)')])

plt.tight_layout()
plt.savefig('../output/fig1_did_main.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig1_did_main.pdf', bbox_inches='tight')
log("  → fig1_did_main saved")
plt.close()



Figure 2: Location heterogeneity

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.1, 4.8))

# A: Response volume
ls = df.groupby('Location').agg(
    resp=('total_responses','sum'), yes=('Yes_Well','sum'),
    rel=('Religion','first')
).reset_index()
ls['prop'] = ls['yes']/ls['resp']
ls = ls.sort_values('resp')

ax1.barh(range(len(ls)), ls['resp'],
         color=['#E09238' if r=='muslim' else '#9238E0' for r in ls['rel']],
         edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(len(ls)))
ax1.set_yticklabels(ls['Location'])
ax1.set_xlabel('Total responses')
ax1.set_xscale('log')
ax1.set_title('A. Response volume', fontweight='bold')
for i,(_, r) in enumerate(ls.iterrows()):
    ax1.text(r['resp']*1.2, i, f"{r['prop']:.2f}", va='center')
ax1.legend(handles=[Patch(fc='#E09238',ec='k',label='Muslim'),
                    Patch(fc='#9238E0',ec='k',label='Christian')],
           loc='lower right')

# B: Before/after gap by location
gaps = []
for loc in df_comm['Location'].unique():
    for opd in [0,1]:
        s = df_comm[(df_comm['Location']==loc)&(df_comm['own_prayer_day']==opd)]
        b = s[s['after_int']==0]; a = s[s['after_int']==1]
        yb,nb = b['Yes_Well'].sum(), b['total_responses'].sum()
        ya,na = a['Yes_Well'].sum(), a['total_responses'].sum()
        if nb>0 and na>0:
            gaps.append({'Location':loc, 'opd':opd, 'gap':ya/na-yb/nb, 'n':nb+na,
                        'rel':df_comm[df_comm['Location']==loc]['Religion'].iloc[0]})
gdf = pd.DataFrame(gaps).sort_values('Location')

for opd, mk, c, lb in [(0,'o','#9238E0',"Other's prayer day"),
                        (1,'s','#E09238','Own prayer day')]:
    g = gdf[gdf['opd']==opd]
    ax2.scatter(g['gap'], g['Location'], marker=mk, color=c,
               s=np.clip(g['n']/8,25,250), alpha=0.7, label=lb,
               edgecolors='black', linewidth=0.5)
ax2.axvline(0, color='grey', ls='--', alpha=0.5)
ax2.set_xlabel('After − Before gap')
ax2.set_title('B. Before/after gap by location', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('../output/fig2_heterogeneity.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig2_heterogeneity.pdf', bbox_inches='tight')
log("  → fig2_heterogeneity saved")
plt.close()

# Ambient Religiosity

In [ ]:
df = df_well_glm.copy()

In [ ]:
df['total_responses'] = df['Yes_Well'] + df['No_Well']

In [ ]:
loc = df.groupby('Location').agg(
    yes=('Yes_Well', 'sum'),
    total=('total_responses', 'sum'),
    pray_ratio=('pray_ratio', 'first'),
    religion=('Religion', 'first'),
    pop_dens=('pop_dens_2025', 'first'),
    nl=('nl_mean_wght', 'first'),
    gdi=('subnational_gdi', 'first'),
    sedac=('sedac_mean', 'first'),
).reset_index()
loc['prop_good'] = loc['yes'] / loc['total']
loc['is_muslim'] = (loc['religion'] == 'muslim').astype(int)
loc['pray_ratio_std'] = (loc['pray_ratio'] - loc['pray_ratio'].mean()) / loc['pray_ratio'].std()

In [ ]:
print("=== Location-level data ===")
print(loc[['Location', 'religion', 'total', 'prop_good', 'pray_ratio']].sort_values('pray_ratio').to_string())
print(f"\nN locations: {len(loc)}")
print(f"Total responses: {loc['total'].sum():,}")

### WLS regressions (weighted by response count)

In [ ]:
# L1: Bivariate
l1 = smf.wls('prop_good ~ pray_ratio_std', data=loc, weights=loc['total']).fit(cov_type='HC1')
print(f"L1 (bivariate): {l1.params['pray_ratio_std']:.4f} ({l1.bse['pray_ratio_std']:.4f}), p={l1.pvalues['pray_ratio_std']:.4f}")
 
# L2: + Religion
l2 = smf.wls('prop_good ~ pray_ratio_std + is_muslim', data=loc, weights=loc['total']).fit(cov_type='HC1')
print(f"L2 (+religion): {l2.params['pray_ratio_std']:.4f} ({l2.bse['pray_ratio_std']:.4f}), p={l2.pvalues['pray_ratio_std']:.4f}")
 
# L3: + SES controls
l3 = smf.wls('prop_good ~ pray_ratio_std + is_muslim + sedac + nl', data=loc, weights=loc['total']).fit(cov_type='HC1')
print(f"L3 (+SES): {l3.params['pray_ratio_std']:.4f} ({l3.bse['pray_ratio_std']:.4f}), p={l3.pvalues['pray_ratio_std']:.4f}")
 
# L4: + all location covariates
l4 = smf.wls('prop_good ~ pray_ratio_std + is_muslim + sedac + nl + pop_dens + gdi', data=loc, weights=loc['total']).fit(cov_type='HC1')
print(f"L4 (+all): {l4.params['pray_ratio_std']:.4f} ({l4.bse['pray_ratio_std']:.4f}), p={l4.pvalues['pray_ratio_std']:.4f}")

### Also unweighted for comparison

In [ ]:
print("\n=== OLS (unweighted) ===")
u1 = smf.ols('prop_good ~ pray_ratio_std', data=loc).fit(cov_type='HC1')
print(f"U1 (bivariate): {u1.params['pray_ratio_std']:.4f} ({u1.bse['pray_ratio_std']:.4f}), p={u1.pvalues['pray_ratio_std']:.4f}")
u2 = smf.ols('prop_good ~ pray_ratio_std + is_muslim', data=loc).fit(cov_type='HC1')
print(f"U2 (+religion): {u2.params['pray_ratio_std']:.4f} ({u2.bse['pray_ratio_std']:.4f}), p={u2.pvalues['pray_ratio_std']:.4f}")
u3 = smf.ols('prop_good ~ pray_ratio_std + is_muslim + sedac + nl', data=loc).fit(cov_type='HC1')
print(f"U3 (+SES): {u3.params['pray_ratio_std']:.4f} ({u3.bse['pray_ratio_std']:.4f}), p={u3.pvalues['pray_ratio_std']:.4f}")
u4 = smf.ols('prop_good ~ pray_ratio_std + is_muslim + sedac + nl + pop_dens + gdi', data=loc).fit(cov_type='HC1')
print(f"U4 (+all): {u4.params['pray_ratio_std']:.4f} ({u4.bse['pray_ratio_std']:.4f}), p={u4.pvalues['pray_ratio_std']:.4f}")

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 11, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.8, 3.25))
 
# --- Panel A: Unconditional scatter ---
for rel, color, marker in [('christian', '#9238E0', 'o'), ('muslim', '#E09238', 's')]:
    sub = loc[loc['religion'] == rel]
    ax1.scatter(sub['pray_ratio'], sub['prop_good'],
                s=np.clip(sub['total'] / 30, 30, 300),
                c=color, marker=marker, alpha=1,
                edgecolors='black', linewidth=0.5,
                label=rel.title())
    for _, row in sub.iterrows():
        ax1.annotate(row['Location'],
                     (row['pray_ratio'], row['prop_good']), alpha=1, ha='left', fontsize=6,
                     xytext=(4, 2), textcoords='offset points')
 
# Overall regression line
b, a = polyfit(loc['pray_ratio'], loc['prop_good'], 1)
x_line = np.linspace(loc['pray_ratio'].min() - 0.01,
                     loc['pray_ratio'].max() + 0.01, 100)
ax1.plot(x_line, a * x_line + b, '--', color='grey', alpha=0.5, linewidth=1)
 
ax1.set_xlabel('Ambient religiosity (prayer ratio)')
ax1.set_ylabel("Proportion reporting 'Good'")
ax1.set_title('A. Location-level association\n(all ads)', fontweight='bold')
ax1.legend(title='Bubble size = N responses')
 
# --- Panel B: Within-religion scatter with separate regression lines ---
for rel, color, marker in [('christian', '#9238E0', 'o'), ('muslim', '#E09238', 's')]:
    sub = loc[loc['religion'] == rel]
    ax2.scatter(sub['pray_ratio'], sub['prop_good'],
                s=np.clip(sub['total'] / 30, 30, 300),
                c=color, marker=marker, alpha=1,
                edgecolors='black', linewidth=0.5,
                label=rel.title())
    if len(sub) >= 3:
        b2, a2 = polyfit(sub['pray_ratio'], sub['prop_good'], 1)
        x_r = np.linspace(sub['pray_ratio'].min() - 0.01,
                          sub['pray_ratio'].max() + 0.01, 50)
        ax2.plot(x_r, a2 * x_r + b2, '--', color=color, alpha=0.5, linewidth=1)
 
ax2.set_xlabel('Ambient religiosity (prayer ratio)')
ax2.set_ylabel("Proportion reporting 'Good'")
ax2.set_title('B. Within-religion association', fontweight='bold')
ax2.legend()
 
plt.tight_layout()
plt.savefig('../output/fig_ambient_religiosity.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig_ambient_religiosity.pdf', bbox_inches='tight')
print('Figure saved')
plt.close()

### LaTeX table

In [ ]:
def sig_stars(p):
    if p < 0.001: return '^{***}'
    if p < 0.01: return '^{**}'
    if p < 0.05: return '^{*}'
    return ''
 
def fmt(model, var):
    if model is None or var not in model.params.index: return '', ''
    c = model.params[var]; se = model.bse[var]; p = model.pvalues[var]
    return f'{c:.4f}{sig_stars(p)}', f'({se:.4f})'

In [ ]:
lines = []
lines.append(r'\begin{table}[!htbp] \centering')
lines.append(r'  \caption{Ambient Religiosity and Well-Being --- Location-Level Analysis}')
lines.append(r'  \label{tab:ambient}')
lines.append(r'\begin{tabular}{@{\extracolsep{5pt}}lcccc}')
lines.append(r'\\[-1.8ex]\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r'& \multicolumn{4}{c}{\textit{Dependent variable: Proportion Good}} \\')
lines.append(r'\cline{2-5}')
lines.append(r'\\[-1.8ex]')
lines.append(r'& (1) & (2) & (3) & (4) \\')
lines.append(r'\hline \\[-1.8ex]')
 
for var, label in [('pray_ratio_std', 'Prayer ratio (std.)'),
                   ('is_muslim', 'Muslim location')]:
    coefs = [fmt(m, var) for m in [l1, l2, l3, l4]]
    c_line = f' {label} & ' + ' & '.join([f'${c}$' if c else '' for c, s in coefs]) + r' \\'
    s_line = ' & ' + ' & '.join([f'${s}$' if s else '' for c, s in coefs]) + r' \\'
    lines.append(c_line)
    lines.append(s_line)
 
lines.append(r'\hline \\[-1.8ex]')
lines.append(r' Religion & No & Yes & Yes & Yes \\')
lines.append(r' Deprivation, nighttime lights & No & No & Yes & Yes \\')
lines.append(r' Population density, GDI & No & No & No & Yes \\')
lines.append(r' Locations & 14 & 14 & 14 & 14 \\')
 
# R-squared
r2s = ' & '.join([f'{m.rsquared:.3f}' for m in [l1, l2, l3, l4]])
lines.append(f' $R^2$ & {r2s}' + r' \\')
 
lines.append(r'\hline')
lines.append(r'\hline \\[-1.8ex]')
note = (r'\multicolumn{5}{l}{\parbox{12cm}{\textit{Note:} $^{*}$p$<$0.05; '
        r'$^{**}$p$<$0.01; $^{***}$p$<$0.001. Weighted OLS at the location level, '
        r'weighted by total response count. HC1 robust standard errors. '
        r'Prayer ratio is the location-level share of respondents reporting daily prayer '
        r'(from the Instagram prayer ad campaign), standardized to mean zero and unit variance. '
        r'The unit of observation is the location ($n = 14$). '
        r'Deprivation is the SEDAC Global Relative Deprivation Index; '
        r'nighttime lights proxy local economic activity; '
        r'GDI is the subnational Gender Development Index.}} \\')
lines.append(note)
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')
 
with open('../output/table_ambient.tex', 'w') as f:
    f.write('\n'.join(lines))
print('Table saved')

# Appendix

In [ ]:
df = df_well_glm.copy()

In [ ]:
df['total_responses'] = df['Yes_Well'] + df['No_Well']
df['after_int'] = df['after'].astype(int)
df['is_muslim'] = (df['Religion'] == 'muslim').astype(int)
df['is_female'] = (df['Gender'] == 'Female').astype(int)
df['age_25_49'] = (df['Age'] == '(25, 49)').astype(int)
df['age_50_65'] = (df['Age'] == '(50, 65)').astype(int)
df['sacred_day'] = 0
df.loc[(df['Religion']=='muslim') & (df['day_of_week_start']=='Friday'), 'sacred_day'] = 1
df.loc[(df['Religion']=='christian') & (df['day_of_week_start']=='Sunday'), 'sacred_day'] = 1
 
weather_covs = ['temperature_2m','precipitation','cloud_cover','wind_speed_10m','relative_humidity_2m']
weather_str = ' + '.join(weather_covs)
demog_str = 'is_female + age_25_49 + age_50_65'
 
df_comm = df[df['day_of_week_start'].isin(['Friday','Sunday'])].copy()
df_tue = df[df['day_of_week_start']=='Tuesday'].copy()

In [ ]:
def get_nonconstant(data, covs):
    return [w for w in covs if data[w].nunique() > 1]

### FIGURE A2: Forest plot of heterogeneity

In [ ]:
# Re-run all subgroup regressions to collect coefficients
results = []

In [ ]:
# Communal DID: Full sample (reference)
m_full = smf.glm(
    f'Yes_Well + No_Well ~ after_int * sacred_day + {demog_str} + {weather_str} + C(Location)',
    data=df_comm, family=sm.families.Binomial()).fit(
    cov_type='cluster', cov_kwds={'groups': df_comm['Location']})
v = 'after_int:sacred_day'
results.append({'label': 'Full sample', 'type': 'Communal',
                'coef': m_full.params[v], 'se': m_full.bse[v],
                'n': df_comm['total_responses'].sum(), 'is_full': True})

In [ ]:
# Communal by gender
for gender_label, gender_val in [('Male', 0), ('Female', 1)]:
    sub = df_comm[df_comm['is_female'] == gender_val].copy()
    m = smf.glm(
        f'Yes_Well + No_Well ~ after_int * sacred_day + age_25_49 + age_50_65 + {weather_str} + C(Location)',
        data=sub, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': sub['Location']})
    results.append({'label': gender_label, 'type': 'Communal',
                    'coef': m.params[v], 'se': m.bse[v],
                    'n': sub['total_responses'].sum(), 'is_full': False})

In [ ]:
# Communal by age
for age_label, age_val in [('Age 18--24', '(18, 24)'), ('Age 25--49', '(25, 49)'), ('Age 50--65', '(50, 65)')]:
    sub = df_comm[df_comm['Age'] == age_val].copy()
    m = smf.glm(
        f'Yes_Well + No_Well ~ after_int * sacred_day + is_female + {weather_str} + C(Location)',
        data=sub, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': sub['Location']})
    results.append({'label': age_label, 'type': 'Communal',
                    'coef': m.params[v], 'se': m.bse[v],
                    'n': sub['total_responses'].sum(), 'is_full': False})

In [ ]:
# Individual DID: Full sample
w_tue = get_nonconstant(df_tue, weather_covs)
m_ind_full = smf.glm(
    f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str} + {" + ".join(w_tue)} + C(Location)',
    data=df_tue, family=sm.families.Binomial()).fit(
    cov_type='cluster', cov_kwds={'groups': df_tue['Location']})
v_ind = 'after_int:is_muslim'
results.append({'label': 'Full sample', 'type': 'Individual',
                'coef': m_ind_full.params[v_ind], 'se': m_ind_full.bse[v_ind],
                'n': df_tue['total_responses'].sum(), 'is_full': True})

In [ ]:
# Individual by gender
for gender_label, gender_val in [('Male', 0), ('Female', 1)]:
    sub = df_tue[df_tue['is_female'] == gender_val].copy()
    w_sub = get_nonconstant(sub, weather_covs)
    try:
        m = smf.glm(
            f'Yes_Well + No_Well ~ after_int * is_muslim + age_25_49 + age_50_65 + {" + ".join(w_sub)} + C(Location)',
            data=sub, family=sm.families.Binomial()).fit(
            cov_type='cluster', cov_kwds={'groups': sub['Location']})
        results.append({'label': gender_label, 'type': 'Individual',
                        'coef': m.params[v_ind], 'se': m.bse[v_ind],
                        'n': sub['total_responses'].sum(), 'is_full': False})
    except:
        results.append({'label': gender_label, 'type': 'Individual',
                        'coef': np.nan, 'se': np.nan,
                        'n': sub['total_responses'].sum(), 'is_full': False})

In [ ]:
# Individual by age
for age_label, age_val in [('Age 18--24', '(18, 24)'), ('Age 25--49', '(25, 49)'), ('Age 50--65', '(50, 65)')]:
    sub = df_tue[df_tue['Age'] == age_val].copy()
    w_sub = get_nonconstant(sub, weather_covs)
    try:
        m = smf.glm(
            f'Yes_Well + No_Well ~ after_int * is_muslim + is_female + {" + ".join(w_sub)} + C(Location)',
            data=sub, family=sm.families.Binomial()).fit(
            cov_type='cluster', cov_kwds={'groups': sub['Location']})
        # Skip if SE is absurdly large (near-separation)
        if m.bse[v_ind] < 5:
            results.append({'label': age_label, 'type': 'Individual',
                            'coef': m.params[v_ind], 'se': m.bse[v_ind],
                            'n': sub['total_responses'].sum(), 'is_full': False})
        else:
            results.append({'label': age_label, 'type': 'Individual',
                            'coef': np.nan, 'se': np.nan,
                            'n': sub['total_responses'].sum(), 'is_full': False})
    except:
        results.append({'label': age_label, 'type': 'Individual',
                        'coef': np.nan, 'se': np.nan,
                        'n': sub['total_responses'].sum(), 'is_full': False})

In [ ]:
rdf = pd.DataFrame(results)

### Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
 
for ax, did_type, title in [(ax1, 'Communal', 'A. Communal Prayer DID'),
                             (ax2, 'Individual', 'B. Individual Prayer DID')]:
    sub = rdf[rdf['type'] == did_type].reset_index(drop=True)
    sub = sub.iloc[::-1].reset_index(drop=True)  # reverse for top-to-bottom
    
    y_positions = range(len(sub))
    
    for i, row in sub.iterrows():
        if np.isnan(row['coef']):
            ax.text(0, i, 'n/a', ha='center', va='center', fontsize=8,
                    color='grey', fontstyle='italic')
            continue
        
        color = '#9238E0' if row['is_full'] else '#E09238'
        marker = 'D' if row['is_full'] else 'o'
        markersize = 8 if row['is_full'] else 6
        linewidth = 2 if row['is_full'] else 1.2
        
        ci_lo = row['coef'] - 1.96 * row['se']
        ci_hi = row['coef'] + 1.96 * row['se']
        
        ax.plot([ci_lo, ci_hi], [i, i], color=color, linewidth=linewidth, solid_capstyle='round')
        ax.plot(row['coef'], i, marker=marker, color=color, markersize=markersize,
                markeredgecolor='black', markeredgewidth=0.5, zorder=5)
    
    ax.axvline(0, color='black', linewidth=0.8, linestyle='-')
    ax.set_yticks(range(len(sub)))
    
    # Labels with sample size
    ylabels = []
    for _, row in sub.iterrows():
        n_str = f" (n={row['n']:,})" if row['n'] < 100000 else f" (n={row['n']/1000:.0f}k)"
        ylabels.append(f"{row['label']}{n_str}")
    ax.set_yticklabels(ylabels, fontsize=9)
    
    ax.set_xlabel('DID interaction coefficient (logit scale)', fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
 
plt.tight_layout()
plt.savefig('../output/fig_appendix_forest.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig_appendix_forest.pdf', bbox_inches='tight')

In [ ]:
z = 1.96 + 0.84
se_communal_lpm = 0.012   # from tab:bootstrap (communal DID, LPM, cluster-robust)
se_individual_lpm = 0.039  # from tab:bootstrap (individual DID, LPM, cluster-robust)
print(f"Communal  MDE = 2.80 * {se_communal_lpm} = {z*se_communal_lpm:.4f}  (~{z*se_communal_lpm*100:.1f} pp)")
print(f"Individual MDE = 2.80 * {se_individual_lpm} = {z*se_individual_lpm:.4f}  (~{z*se_individual_lpm*100:.1f} pp)")

In [ ]:
df = df_well_glm.copy()
df['total_responses'] = df['Yes_Well'] + df['No_Well']
df['prop_good'] = df['Yes_Well'] / df['total_responses']
df['after_int'] = df['after'].astype(int)
df['is_muslim'] = (df['Religion'] == 'muslim').astype(int)
df['is_female'] = (df['Gender'] == 'Female').astype(int)
df['age_25_49'] = (df['Age'] == '(25, 49)').astype(int)
df['age_50_65'] = (df['Age'] == '(50, 65)').astype(int)
df['sacred_day'] = 0
df.loc[(df['Religion']=='muslim') & (df['day_of_week_start']=='Friday'), 'sacred_day'] = 1
df.loc[(df['Religion']=='christian') & (df['day_of_week_start']=='Sunday'), 'sacred_day'] = 1
 
weather_covs = ['temperature_2m','precipitation','cloud_cover','wind_speed_10m','relative_humidity_2m']
weather_str = ' + '.join(weather_covs)
demog_str = 'is_female + age_25_49 + age_50_65'
 
df_comm = df[df['day_of_week_start'].isin(['Friday','Sunday'])].copy()
df_tue = df[df['day_of_week_start']=='Tuesday'].copy()

In [ ]:
def get_nonconstant(data, covs):
    return [w for w in covs if data[w].nunique() > 1]

### POST-HOC POWER ANALYSIS

In [ ]:
# For the communal DID: what is the minimum detectable effect (MDE)?
# The DID is estimated on grouped binomial data.
# We approximate power using the normal approximation for proportions.
#
# Key parameters:
# - Total responses in the communal sample
# - Proportion in each of the 4 DID cells
# - Desired power (80%) and significance level (5%)

In [ ]:
# Cell sizes (from the analysis)
cells = {}
for sd in [0, 1]:
    for aft in [0, 1]:
        s = df_comm[(df_comm['sacred_day']==sd) & (df_comm['after_int']==aft)]
        y = s['Yes_Well'].sum()
        n = s['total_responses'].sum()
        cells[(sd, aft)] = {'y': y, 'n': n, 'p': y/n if n > 0 else 0}

In [ ]:
# Communal DID cell sizes
for (sd, aft), v in cells.items():
    print(f"  sacred_day={sd}, after={aft}: n={v['n']:,}, p={v['p']:.4f}")

In [ ]:
# Overall baseline proportion
p_bar = df_comm['Yes_Well'].sum() / df_comm['total_responses'].sum()
print(f"\nOverall baseline proportion: {p_bar:.4f}")

In [ ]:
# MDE calculation for a 2x2 DID on proportions
# Using the formula: MDE = (z_alpha + z_beta) * SE(DID)
# where SE(DID) = sqrt(p*(1-p) * (1/n00 + 1/n01 + 1/n10 + 1/n11))
# This ignores clustering but gives a lower bound on MDE

In [ ]:
z_alpha = 1.96  # two-sided 5%
z_beta = 0.84   # 80% power

In [ ]:
n00 = cells[(0,0)]['n']
n01 = cells[(0,1)]['n']
n10 = cells[(1,0)]['n']
n11 = cells[(1,1)]['n']
 
se_did = np.sqrt(p_bar * (1-p_bar) * (1/n00 + 1/n01 + 1/n10 + 1/n11))
mde_naive = (z_alpha + z_beta) * se_did

print(f"  SE(DID) = {se_did:.4f}")
print(f"  MDE (80% power, 5% sig) = {mde_naive:.4f} ({mde_naive*100:.2f} pp)")

In [ ]:
# With clustering: inflate SE by design effect
# Approximate design effect from the model
# Use the ratio of clustered SE to naive SE from our preferred model

In [ ]:
m_naive = smf.glm(
    f'Yes_Well + No_Well ~ after_int * sacred_day + {demog_str} + {weather_str} + C(Location)',
    data=df_comm, family=sm.families.Binomial()).fit()
m_clust = smf.glm(
    f'Yes_Well + No_Well ~ after_int * sacred_day + {demog_str} + {weather_str} + C(Location)',
    data=df_comm, family=sm.families.Binomial()).fit(
    cov_type='cluster', cov_kwds={'groups': df_comm['Location']})
 
v = 'after_int:sacred_day'
se_naive_model = m_naive.bse[v]
se_clust_model = m_clust.bse[v]
deff = (se_clust_model / se_naive_model) ** 2
 


In [ ]:
# Design effect from clustering
print(f"  SE(naive model) = {se_naive_model:.4f}")
print(f"  SE(clustered)   = {se_clust_model:.4f}")
print(f"  Design effect   = {deff:.2f}")

In [ ]:
mde_clustered = mde_naive * np.sqrt(deff)

In [ ]:
# Cluster-adjusted MDE ---")
print(f"  MDE (80% power, 5% sig) = {mde_clustered:.4f} ({mde_clustered*100:.2f} pp)")

In [ ]:
# Also compute for individual DID
# Individual DID (Tuesday):
w_tue_covs = get_nonconstant(df_tue, weather_covs)
m_ind_clust = smf.glm(
    f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str} + {" + ".join(w_tue_covs)} + C(Location)',
    data=df_tue, family=sm.families.Binomial()).fit(
    cov_type='cluster', cov_kwds={'groups': df_tue['Location']})
m_ind_naive = smf.glm(
    f'Yes_Well + No_Well ~ after_int * is_muslim + {demog_str} + {" + ".join(w_tue_covs)} + C(Location)',
    data=df_tue, family=sm.families.Binomial()).fit()
 
v_ind = 'after_int:is_muslim'
se_ind_clust = m_ind_clust.bse[v_ind]
se_ind_naive = m_ind_naive.bse[v_ind]
deff_ind = (se_ind_clust / se_ind_naive) ** 2
 
p_bar_tue = df_tue['Yes_Well'].sum() / df_tue['total_responses'].sum()
cells_tue = {}
for mus in [0, 1]:
    for aft in [0, 1]:
        s = df_tue[(df_tue['is_muslim']==mus) & (df_tue['after_int']==aft)]
        cells_tue[(mus, aft)] = s['total_responses'].sum()
 
se_did_tue = np.sqrt(p_bar_tue*(1-p_bar_tue) * sum(1/n for n in cells_tue.values()))
mde_ind = (z_alpha + z_beta) * se_did_tue * np.sqrt(deff_ind)
print(f"  Design effect = {deff_ind:.2f}")
print(f"  MDE = {mde_ind:.4f} ({mde_ind*100:.2f} pp)")

In [ ]:
# Convert logit-scale MDE to probability scale for interpretation
# At p=0.5, d(logit)/dp = 4, so delta_p ≈ delta_logit / 4
mde_prob_comm = mde_clustered

In [ ]:
# Summary
print(f"  Communal DID MDE (probability scale, approx): {mde_prob_comm:.4f} ({mde_prob_comm*100:.2f} pp)")
print(f"  Communal DID MDE (logit scale): {mde_clustered:.4f}")
print(f"  Individual DID MDE (logit scale): {mde_ind:.4f}")

### HETEROGENEITY BY GENDER AND AGE

In [ ]:
# Communal DID by gender
gender_results = []
for gender_label, gender_val in [('Male', 0), ('Female', 1)]:
    sub = df_comm[df_comm['is_female'] == gender_val].copy()
    m = smf.glm(
        f'Yes_Well + No_Well ~ after_int * sacred_day + age_25_49 + age_50_65 + {weather_str} + C(Location)',
        data=sub, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': sub['Location']})
    v = 'after_int:sacred_day'
    p = m.pvalues[v]
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
    print(f"  {gender_label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}, "
          f"N_ads={int(m.nobs)}, N_resp={sub['total_responses'].sum():,}")
    gender_results.append({
        'Subgroup': gender_label, 'DID_type': 'Communal',
        'coef': m.params[v], 'se': m.bse[v], 'p': p,
        'n_ads': int(m.nobs), 'n_resp': sub['total_responses'].sum()
    })

In [ ]:
# Communal DID by age
age_results = []
for age_label, age_col_val in [('18-24', '(18, 24)'), ('25-49', '(25, 49)'), ('50-65', '(50, 65)')]:
    sub = df_comm[df_comm['Age'] == age_col_val].copy()
    m = smf.glm(
        f'Yes_Well + No_Well ~ after_int * sacred_day + is_female + {weather_str} + C(Location)',
        data=sub, family=sm.families.Binomial()).fit(
        cov_type='cluster', cov_kwds={'groups': sub['Location']})
    v = 'after_int:sacred_day'
    p = m.pvalues[v]
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
    print(f"  Age {age_label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}, "
          f"N_ads={int(m.nobs)}, N_resp={sub['total_responses'].sum():,}")
    age_results.append({
        'Subgroup': f'Age {age_label}', 'DID_type': 'Communal',
        'coef': m.params[v], 'se': m.bse[v], 'p': p,
        'n_ads': int(m.nobs), 'n_resp': sub['total_responses'].sum()
    })

In [ ]:
# Individual DID by gender
for gender_label, gender_val in [('Male', 0), ('Female', 1)]:
    sub = df_tue[df_tue['is_female'] == gender_val].copy()
    w_sub = get_nonconstant(sub, weather_covs)
    try:
        m = smf.glm(
            f'Yes_Well + No_Well ~ after_int * is_muslim + age_25_49 + age_50_65 + {" + ".join(w_sub)} + C(Location)',
            data=sub, family=sm.families.Binomial()).fit(
            cov_type='cluster', cov_kwds={'groups': sub['Location']})
        v = 'after_int:is_muslim'
        p = m.pvalues[v]
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
        print(f"  {gender_label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}, "
              f"N_ads={int(m.nobs)}, N_resp={sub['total_responses'].sum():,}")
        gender_results.append({
            'Subgroup': gender_label, 'DID_type': 'Individual',
            'coef': m.params[v], 'se': m.bse[v], 'p': p,
            'n_ads': int(m.nobs), 'n_resp': sub['total_responses'].sum()
        })
    except Exception as e:
        print(f"  {gender_label}: FAILED ({e})")
        gender_results.append({
            'Subgroup': gender_label, 'DID_type': 'Individual',
            'coef': np.nan, 'se': np.nan, 'p': np.nan,
            'n_ads': len(sub), 'n_resp': sub['total_responses'].sum()
        })

In [ ]:
# Individual DID by age
for age_label, age_col_val in [('18-24', '(18, 24)'), ('25-49', '(25, 49)'), ('50-65', '(50, 65)')]:
    sub = df_tue[df_tue['Age'] == age_col_val].copy()
    w_sub = get_nonconstant(sub, weather_covs)
    try:
        m = smf.glm(
            f'Yes_Well + No_Well ~ after_int * is_muslim + is_female + {" + ".join(w_sub)} + C(Location)',
            data=sub, family=sm.families.Binomial()).fit(
            cov_type='cluster', cov_kwds={'groups': sub['Location']})
        v = 'after_int:is_muslim'
        p = m.pvalues[v]
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
        print(f"  Age {age_label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}, "
              f"N_ads={int(m.nobs)}, N_resp={sub['total_responses'].sum():,}")
        age_results.append({
            'Subgroup': f'Age {age_label}', 'DID_type': 'Individual',
            'coef': m.params[v], 'se': m.bse[v], 'p': p,
            'n_ads': int(m.nobs), 'n_resp': sub['total_responses'].sum()
        })
    except Exception as e:
        print(f"  Age {age_label}: FAILED ({e})")
        age_results.append({
            'Subgroup': f'Age {age_label}', 'DID_type': 'Individual',
            'coef': np.nan, 'se': np.nan, 'p': np.nan,
            'n_ads': len(sub), 'n_resp': sub['total_responses'].sum()
        })

In [ ]:
all_hetero = gender_results + age_results

### RESPONSE RATE AS OUTCOME

In [ ]:
# Communal DID on response rate
m_rr1 = smf.ols(f'well_response ~ after_int * sacred_day + {demog_str} + {weather_str}',
                data=df_comm).fit(cov_type='cluster', cov_kwds={'groups': df_comm['Location']})
m_rr2 = smf.ols(f'well_response ~ after_int * sacred_day + {demog_str} + {weather_str} + C(Location)',
                data=df_comm).fit(cov_type='cluster', cov_kwds={'groups': df_comm['Location']})
 
for label, m in [('No loc FE', m_rr1), ('With loc FE', m_rr2)]:
    v = 'after_int:sacred_day'
    p = m.pvalues[v]
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
    print(f"  {label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}")

In [ ]:
# Individual DID on response rate
m_rr3 = smf.ols(f'well_response ~ after_int * is_muslim + {demog_str} + {" + ".join(get_nonconstant(df_tue, weather_covs))}',
                data=df_tue).fit(cov_type='cluster', cov_kwds={'groups': df_tue['Location']})
m_rr4 = smf.ols(f'well_response ~ after_int * is_muslim + {demog_str} + {" + ".join(get_nonconstant(df_tue, weather_covs))} + C(Location)',
                data=df_tue).fit(cov_type='cluster', cov_kwds={'groups': df_tue['Location']})
 
for label, m in [('No loc FE', m_rr3), ('With loc FE', m_rr4)]:
    v = 'after_int:is_muslim'
    p = m.pvalues[v]
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
    print(f"  {label}: coef={m.params[v]:.4f} ({m.bse[v]:.4f}), p={p:.4f} {sig}")

In [ ]:
# Descriptive: mean response rate before vs after
for label, sub in [('Communal (all)', df_comm), ('Tue Muslim', df_tue[df_tue['is_muslim']==1]),
                   ('Tue Christian', df_tue[df_tue['is_muslim']==0])]:
    before = sub[sub['after_int']==0]['well_response'].mean()
    after = sub[sub['after_int']==1]['well_response'].mean()
    print(f"  {label}: before={before:.3f}%, after={after:.3f}%, diff={after-before:+.3f} pp")

### GENERATE LATEX APPENDIX

In [ ]:
def sig_stars(p):
    if np.isnan(p): return ''
    if p < 0.001: return '^{***}'
    if p < 0.01: return '^{**}'
    if p < 0.05: return '^{*}'
    return ''

In [ ]:
lines = []
lines.append(r'% =============================================================')
lines.append(r'% APPENDIX')
lines.append(r'% =============================================================')
lines.append(r'')
lines.append(r'\appendix')
lines.append(r'\section{Additional Analyses}\label{app:additional}')
lines.append(r'')
 
# --- A1: Power analysis ---
lines.append(r'\subsection{Post-hoc power analysis}\label{app:power}')
lines.append(r'')
lines.append(r'To assess whether our null findings reflect genuine absence of effects or insufficient statistical power, we compute minimum detectable effect sizes (MDE) for the two Instagram DID designs at 80\% power and 5\% significance (two-sided).')
lines.append(r'')
lines.append(f'For the communal prayer DID, the four cells of the 2$\\times$2 design contain {n00:,}, {n01:,}, {n10:,}, and {n11:,} responses, respectively, with an overall baseline proportion of {p_bar:.3f}. Ignoring clustering, the MDE on the logit scale is {mde_naive:.3f}. Adjusting for the estimated design effect from location-level clustering (DEFF $= {deff:.1f}$), the cluster-adjusted MDE is {mde_clustered:.3f} on the logit scale, corresponding to approximately {mde_prob_comm*100:.1f} percentage points on the probability scale. This means our design can reliably detect shifts in the proportion reporting ``Good\'\' of roughly {mde_prob_comm*100:.0f} percentage points or larger, but would miss smaller effects.')
lines.append(r'')
lines.append(f'For the individual prayer DID, the design effect is larger (DEFF $= {deff_ind:.1f}$) due to greater between-location heterogeneity on Tuesdays, yielding an MDE of {mde_ind:.3f} on the logit scale.')
lines.append(r'')
lines.append(r'These MDEs are modest relative to the raw (pre-fixed-effects) DID estimates of 0.128 and 0.136, suggesting that our design has adequate power to detect effects of the magnitude observed in the simple specifications. The null results in the preferred specifications with location fixed effects are therefore unlikely to reflect pure power limitations; rather, they indicate that the raw associations are absorbed by cross-location heterogeneity.')
lines.append(r'')
lines.append(r'')
 
# --- A2: Heterogeneity table ---
lines.append(r'\subsection{Heterogeneity by gender and age}\label{app:heterogeneity}')
lines.append(r'')
lines.append(r'Table~\ref{tab:heterogeneity} reports the DID interaction estimated separately by gender and age group. All specifications include the full set of controls, weather covariates, and location fixed effects. These subgroup analyses are exploratory and should be interpreted cautiously given the reduced sample sizes.')
lines.append(r'')
 
lines.append(r'\begin{table}[!htbp] \centering')
lines.append(r'  \caption{Heterogeneity in Prayer DID by Gender and Age}')
lines.append(r'  \label{tab:heterogeneity}')
lines.append(r'\begin{tabular}{@{\extracolsep{5pt}}llrrrr}')
lines.append(r'\\[-1.8ex]\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r' Subgroup & DID type & Coefficient & SE & $p$-value & Ads \\')
lines.append(r'\hline \\[-1.8ex]')
 
for row in all_hetero:
    coef_str = f"${row['coef']:.4f}{sig_stars(row['p'])}$" if not np.isnan(row['coef']) else '---'
    se_str = f"$({row['se']:.4f})$" if not np.isnan(row['se']) else ''
    p_str = f"{row['p']:.4f}" if not np.isnan(row['p']) else '---'
    lines.append(f" {row['Subgroup']} & {row['DID_type']} & {coef_str} & {se_str} & {p_str} & {row['n_ads']} \\\\")
 
lines.append(r'\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r'\multicolumn{6}{l}{\parbox{12cm}{\textit{Note:} $^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. Each row reports the DID interaction coefficient from a separate binomial GLM estimated on the indicated subgroup, with weather controls and location fixed effects. Standard errors clustered at the location level. These analyses are exploratory; reduced sample sizes limit statistical power for subgroup comparisons.}} \\')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')
lines.append(r'')
lines.append(r'')
 
# --- A3: Response rate as outcome ---
lines.append(r'\subsection{Response rate as outcome}\label{app:response_rate}')
lines.append(r'')
lines.append(r'The main analysis examines whether prayer affects the \textit{content} of responses (the proportion reporting ``Good\'\'). A distinct question is whether prayer affects \textit{engagement}: do more people respond to the ad after a prayer event? If prayer increases the propensity to engage with a well-being question on social media, this would constitute a behavioral effect even if the reported well-being level does not change.')
lines.append(r'')
lines.append(r'Table~\ref{tab:response_rate_did} reports the DID interaction with the ad-level response rate (total responses divided by reach, in percent) as the outcome, estimated by OLS with location-clustered standard errors.')
lines.append(r'')
 
# Response rate DID table
lines.append(r'\begin{table}[!htbp] \centering')
lines.append(r'  \caption{Prayer DID with Response Rate as Outcome}')
lines.append(r'  \label{tab:response_rate_did}')
lines.append(r'\begin{tabular}{@{\extracolsep{5pt}}lcccc}')
lines.append(r'\\[-1.8ex]\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r'& \multicolumn{2}{c}{Communal DID} & \multicolumn{2}{c}{Individual DID} \\')
lines.append(r'\cline{2-3} \cline{4-5}')
lines.append(r'\\[-1.8ex]')
lines.append(r'& (1) & (2) & (3) & (4) \\')
lines.append(r'\hline \\[-1.8ex]')
 
# After
for v_comm, v_ind, label in [
    ('after_int', 'after_int', 'After'),
    ('sacred_day', 'is_muslim', 'Sacred day / Muslim loc.'),
    ('after_int:sacred_day', 'after_int:is_muslim', 'After $\\times$ Treatment'),
]:
    vals = []
    for m, v in [(m_rr1, v_comm), (m_rr2, v_comm), (m_rr3, v_ind), (m_rr4, v_ind)]:
        if v in m.params.index:
            p = m.pvalues[v]
            c = m.params[v]
            se = m.bse[v]
            if se > 100:
                vals.append(('', ''))
            else:
                vals.append((f'{c:.4f}{sig_stars(p)}', f'({se:.4f})'))
        else:
            vals.append(('', ''))
 
    c_line = f' {label} & ' + ' & '.join([f'${c}$' if c else '' for c, s in vals]) + r' \\'
    s_line = ' & ' + ' & '.join([f'${s}$' if s else '' for c, s in vals]) + r' \\'
    lines.append(c_line)
    lines.append(s_line)
 
lines.append(r'\hline \\[-1.8ex]')
lines.append(r' Controls & Yes & Yes & Yes & Yes \\')
lines.append(r' Location FE & No & Yes & No & Yes \\')
lines.append(f' Observations & {int(m_rr1.nobs)} & {int(m_rr2.nobs)} & {int(m_rr3.nobs)} & {int(m_rr4.nobs)}' + r' \\')
lines.append(r'\hline')
lines.append(r'\hline \\[-1.8ex]')
lines.append(r'\multicolumn{5}{l}{\parbox{11cm}{\textit{Note:} $^{*}$p$<$0.05; $^{**}$p$<$0.01; $^{***}$p$<$0.001. OLS with standard errors clustered at the location level. The dependent variable is the ad-level response rate (total poll responses / reach $\times$ 100). Columns (1)--(2) use Friday and Sunday ads; columns (3)--(4) use Tuesday ads. All specifications include demographic and weather controls.}} \\')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')
lines.append(r'')
 
# Closing paragraph for response rate section
lines.append(r'Across all specifications, the DID interaction on response rate is statistically insignificant, suggesting that prayer events do not detectably shift ad engagement. This complements the main finding: prayer affects neither the propensity to respond nor the content of the response in our data.')
lines.append(r'')

### Appendix: Performance Analysis

In [ ]:
_rf = pd.concat([wellbeing_1[['ID','Total_Poll','Yes','No']],
                 wellbeing_2[['ID','Total_Poll','Yes','No']]], ignore_index=True)
_rf = _rf.replace(',','',regex=True).astype('int64').set_index('ID')
_wb = control_vars['Gender'].isin(['Male','Female'])
for _c in ['Total_Poll','Yes','No']:
    control_vars.loc[_wb, _c] = (control_vars.loc[_wb,'ID'].map(_rf[_c])
                                 .fillna(control_vars.loc[_wb,_c]).astype('int64'))

df = control_vars.copy()
df['response_rate'] = np.where(df['reach'] > 0, df['Total_Poll'] / df['reach'] * 100, 0)
df['cost_per_response'] = np.where(df['Total_Poll'] > 0, df['spend'] / df['Total_Poll'], np.nan)
df['prop_good'] = np.where(df['Total_Poll'] > 0, df['Yes'] / df['Total_Poll'], np.nan)
df['completion_75'] = np.where(df['impressions'] > 0, df['video_p75_watched'] / df['impressions'] * 100, 0)
df['freq'] = np.where(df['reach'] > 0, df['impressions'] / df['reach'], np.nan)

In [ ]:
df_wb = df[df['Gender'].isin(['Male', 'Female'])].copy()
df_pr = df[df['Gender'].isin(['Islam', 'Christian'])].copy()

In [ ]:
df_wb.spend.sum()/df_wb.Total_Poll.sum()

In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 11, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9})

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11.5, 10.0))
 
# ── Panel A: Spend vs responses by location ──
ax = axes[0, 0]
loc_agg = df_wb.groupby('Location').agg(
    spend=('spend', 'sum'),
    responses=('Total_Poll', 'sum'),
    religion=('Religion', 'first')
).reset_index()
colors = ['#9238E0' if r == 'christian' else '#E09238' for r in loc_agg['religion']]
ax.scatter(loc_agg['spend'], loc_agg['responses'], c=colors, s=60,
           edgecolors='black', linewidth=0.5, alpha=1)
_topA = loc_agg.nlargest(3, 'responses')['Location'].tolist()
for _, row in loc_agg.iterrows():
    if row['Location'] in _topA:
        ax.annotate(row['Location'], (row['spend'], row['responses']), alpha=0.7, xytext=(3, 2), textcoords='offset points')
ax.set_xlabel('Total spend (EUR)')
ax.set_ylabel('Total poll responses')
ax.set_title('A. Spend vs. responses\nby location', fontweight='bold')
ax.legend(handles=[Patch(fc='#9238E0', label='Christian'),
                   Patch(fc='#E09238', label='Muslim')])
 
# ── Panel B: Response rate by hour of day ──
ax = axes[0, 1]
df_wb_r = df_wb[df_wb['reach'] > 0].copy()
hourly = df_wb_r.groupby('hour_of_day_avg').agg(
    mean_rr=('response_rate', 'mean'),
    se_rr=('response_rate', lambda x: x.std() / np.sqrt(len(x))),
    n=('response_rate', 'count')
).reset_index()
ax.bar(hourly['hour_of_day_avg'], hourly['mean_rr'], width=0.7,
       color='#38E092', edgecolor='black', linewidth=0.5, alpha=1)
ax.errorbar(hourly['hour_of_day_avg'], hourly['mean_rr'],
            yerr=1.96 * hourly['se_rr'], fmt='none', color='black', capsize=3)
for _, row in hourly.iterrows():
    ax.text(row['hour_of_day_avg'], row['mean_rr'] + 1.96 * row['se_rr'] + 0.02,
            f"n={int(row['n'])}", ha='center')
ax.set_xlabel('Hour of day (local time)')
ax.set_ylabel('Response rate (%)')
ax.set_title('B. Response rate\nby hour of day', fontweight='bold')
ax.set_xticks(hourly['hour_of_day_avg'])
 
# ── Panel C: Response rate by day of week ──
ax = axes[0, 2]
day_order = ['Tuesday', 'Friday', 'Sunday']
day_colors = {'Tuesday': '#707070', 'Friday': '#E09238', 'Sunday': '#9238E0'}
daily = df_wb_r[df_wb_r['day_of_week_start'].isin(day_order)].groupby('day_of_week_start').agg(
    mean_rr=('response_rate', 'mean'),
    se_rr=('response_rate', lambda x: x.std() / np.sqrt(len(x))),
    n=('response_rate', 'count')
).reindex(day_order)
bars = ax.bar(range(len(day_order)), daily['mean_rr'], width=0.6,
              color=[day_colors[d] for d in day_order],
              edgecolor='black', linewidth=0.5, alpha=1)
ax.errorbar(range(len(day_order)), daily['mean_rr'],
            yerr=1.96 * daily['se_rr'], fmt='none', color='black', capsize=4)
ax.set_xticks(range(len(day_order)))
ax.set_xticklabels(['Tue', 'Fri', 'Sun'])
ax.set_ylabel('Response rate (%)')
ax.set_title('C. Response rate\nby day of week', fontweight='bold')
 
# ── Panel D: Cost per response distribution ──
ax = axes[1, 0]
cpr = df_wb[df_wb['Total_Poll'] > 0]['cost_per_response'].dropna()
ax.hist(cpr[cpr < 2], bins=30, color='#38E092', edgecolor='black',
        linewidth=0.5, alpha=1)
weighted_cpr = df_wb['spend'].sum() / df_wb['Total_Poll'].sum()
ax.axvline(weighted_cpr, color='red', ls='--', linewidth=1, label=f'Aggregate: {weighted_cpr:.2f}')
ax.axvline(cpr.median(), color='black', ls='--', linewidth=1, label=f'Median: {cpr.median():.2f}')
ax.set_xlabel('Cost per response (EUR)')
ax.set_ylabel('Count (ads)')
ax.set_title('D. Cost per poll response\n(wellbeing ads)', fontweight='bold')
ax.legend(loc='upper right')
 
# ── Panel E: Video completion rate vs response rate ──
ax = axes[1, 1]
sub = df_wb[(df_wb['impressions'] > 0) & (df_wb['reach'] > 0) & (df_wb['Total_Poll'] > 0)].copy()
ax.scatter(sub['completion_75'], sub['response_rate'],
           s=15, alpha=0.4, c='#636363', edgecolors='none')
# Add correlation
from scipy.stats import pearsonr
r, p = pearsonr(sub['completion_75'], sub['response_rate'])
ax.text(0.95, 0.95, f'r = {r:.2f} (p = {p:.3f})',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xlabel('Video 75% completion rate (%)')
ax.set_ylabel('Poll response rate (%)')
ax.set_title('E. Video engagement vs.\npoll response', fontweight='bold')
 
# ── Panel F: Reach vs response rate by location ──
ax = axes[1, 2]
loc_rr = df_wb[df_wb['reach'] > 0].groupby('Location').agg(
    total_reach=('reach', 'sum'),
    total_poll=('Total_Poll', 'sum'),
    religion=('Religion', 'first')
).reset_index()
loc_rr['agg_response_rate'] = loc_rr['total_poll'] / loc_rr['total_reach'] * 100
colors_rr = ['#9238E0' if r == 'christian' else '#E09238' for r in loc_rr['religion']]
ax.scatter(loc_rr['total_reach'], loc_rr['agg_response_rate'],
           c=colors_rr, s=60, edgecolors='black', linewidth=0.5, alpha=1)
_topF = loc_rr.nlargest(2, 'total_reach')['Location'].tolist()
for _, row in loc_rr.iterrows():
    if row['Location'] in _topF:
        ax.annotate(row['Location'], (row['total_reach'], row['agg_response_rate']), alpha=0.7, xytext=(3, 2), textcoords='offset points')
ax.set_xlabel('Total reach')
ax.set_ylabel('Aggregate response rate (%)')
ax.set_title('F. Reach vs. response rate\nby location', fontweight='bold')
ax.set_xscale('log')

#### Cost Efficiency figures

wb_color = '#9238E0'
pr_color = '#E09238'
 
# ── G: Paired dot plot — response rate by location ──
ax = axes[2, 0]
 
wb_loc = df_wb.groupby('Location').agg(
    reach=('reach', 'sum'), poll=('Total_Poll', 'sum'),
    religion=('Religion', 'first')
).reset_index()
wb_loc['rr'] = wb_loc['poll'] / wb_loc['reach'] * 100
 
pr_loc = df_pr.groupby('Location').agg(
    reach=('reach', 'sum'), poll=('Total_Poll', 'sum'),
    religion=('Religion', 'first')
).reset_index()
pr_loc['rr'] = pr_loc['poll'] / pr_loc['reach'] * 100
 
paired = wb_loc.merge(
    pr_loc[['Location', 'rr']], on='Location', suffixes=('_wb', '_pr')
).sort_values('rr_wb')
 
for _, row in paired.iterrows():
    color = '#9238E0' if row['religion'] == 'christian' else '#E09238'
    ax.plot([0, 1], [row['rr_wb'], row['rr_pr']], color=color, alpha=0.5,
            linewidth=1.2, zorder=1)
    ax.scatter(0, row['rr_wb'], c=color, s=30, edgecolors='black',
               linewidth=0.4, zorder=3)
    ax.scatter(1, row['rr_pr'], c=color, s=30, edgecolors='black',
               linewidth=0.4, zorder=3)
 
# Aggregate
wb_agg_rr = df_wb['Total_Poll'].sum() / df_wb['reach'].sum() * 100
pr_agg_rr = df_pr['Total_Poll'].sum() / df_pr['reach'].sum() * 100
ax.plot([0, 1], [wb_agg_rr, pr_agg_rr], color='black', linewidth=2,
        linestyle='--', zorder=2, alpha=0.7)
ax.scatter([0, 1], [wb_agg_rr, pr_agg_rr], c='black', s=60, marker='D',
           edgecolors='black', linewidth=0.5, zorder=4)
 
ax.set_xticks([0, 1])
ax.set_xticklabels(['Well-being\n(~2 hrs)', 'Prayer\n(1 week)'])
ax.set_ylabel('Response rate (%)')
ax.set_title('G. Response rate by location\n(well-being vs. prayer)', fontweight='bold')
ax.set_xlim(-0.3, 1.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(handles=[Patch(fc='#9238E0', ec='k', label='Christian'),
                   Patch(fc='#E09238', ec='k', label='Muslim'),
                   plt.Line2D([0], [0], color='black', ls='--', marker='D',
                              markersize=5, label='Aggregate')], loc='upper left')
 
# ── H: Reach per ad ──
ax = axes[2, 1]
categories = ['Well-being\n(~2 hrs)', 'Prayer\n(1 week)']
wb_reach_mean = df_wb['reach'].mean()
pr_reach_mean = df_pr['reach'].mean()
wb_reach_se = df_wb['reach'].std() / np.sqrt(len(df_wb))
pr_reach_se = df_pr['reach'].std() / np.sqrt(len(df_pr))
bars = ax.bar([0, 1], [wb_reach_mean, pr_reach_mean], width=0.6,
              color=[wb_color, pr_color], edgecolor='black', linewidth=0.5, alpha=1)
ax.errorbar([0, 1], [wb_reach_mean, pr_reach_mean],
            yerr=[1.96 * wb_reach_se, 1.96 * pr_reach_se],
            fmt='none', color='black', capsize=5)
# Add text labels
_h_top = max(wb_reach_mean + 1.96 * wb_reach_se, pr_reach_mean + 1.96 * pr_reach_se)
ax.set_ylim(0, _h_top * 1.45)
ax.text(0, wb_reach_mean + 1.96 * wb_reach_se + _h_top * 0.04,
        f'{wb_reach_mean:,.0f}\n(n={len(df_wb)})', ha='center', va='bottom')
ax.text(1, pr_reach_mean + 1.96 * pr_reach_se + _h_top * 0.04,
        f'{pr_reach_mean:,.0f}\n(n={len(df_pr)})', ha='center', va='bottom')
ax.set_xticks([0, 1])
ax.set_xticklabels(categories)
ax.set_ylabel('Mean reach per ad')
ax.set_title('H. Reach per ad\n(well-being vs. prayer)', fontweight='bold')
 
# ── I: Cost efficiency comparison ──
ax = axes[2, 2]
# Bar chart: aggregate cost/response, cost/1000 reached, responses/EUR
metrics = {
    'Cost/resp. (EUR)': (
        df_wb['spend'].sum() / df_wb['Total_Poll'].sum(),
        df_pr['spend'].sum() / df_pr['Total_Poll'].sum()
    ),
    'Cost/1k reached (EUR)': (
        df_wb['spend'].sum() / df_wb['reach'].sum() * 1000,
        df_pr['spend'].sum() / df_pr['reach'].sum() * 1000
    ),
    'Resp./EUR spent': (
        df_wb['Total_Poll'].sum() / df_wb['spend'].sum(),
        df_pr['Total_Poll'].sum() / df_pr['spend'].sum()
    ),
}
 
x = np.arange(len(metrics))
width = 0.35
wb_vals = [v[0] for v in metrics.values()]
pr_vals = [v[1] for v in metrics.values()]
 
bars1 = ax.bar(x - width/2, wb_vals, width, color=wb_color,
               edgecolor='black', linewidth=0.5, alpha=1, label='Well-being')
bars2 = ax.bar(x + width/2, pr_vals, width, color=pr_color,
               edgecolor='black', linewidth=0.5, alpha=1, label='Prayer')
 
# Add value labels
for bar_group in [bars1, bars2]:
    for bar in bar_group:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f'{h:.2f}', ha='center', va='bottom')
 
ax.set_ylim(0, max(wb_vals + pr_vals) * 1.20)
ax.set_xticks(x)
ax.set_xticklabels(list(metrics.keys()), rotation=20, ha='right')
ax.set_title('I. Cost efficiency\n(well-being vs. prayer)', fontweight='bold')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('../output/fig_ad_performance.png', dpi=300, bbox_inches='tight')
plt.savefig('../output/fig_ad_performance.pdf', bbox_inches='tight')


In [ ]:
print(f"\nKey stats:")
print(f"  Total actual spend (wellbeing): EUR {df_wb['spend'].sum():.2f}")
print(f"  Total actual spend (prayer): EUR {df_pr['spend'].sum():.2f}")
print(f"  Total actual spend (all): EUR {df['spend'].sum():.2f}")
print(f"  Median cost/response (wellbeing): EUR {cpr.median():.4f}")
print(f"  Mean cost/response (wellbeing): EUR {cpr.mean():.4f}")
print(f"  Correlation (video completion, response rate): r={r:.2f}, p={p:.3f}")